### 병해충 데이터 추출

In [ ]:
# -*- coding: utf-8 -*-
import os, re, glob
import pandas as pd

# ================== 설정 ==================
INPUT_DIR = "/Users/doyoung-gil/연구실/데이터/관찰데이터+지점ID+datetime"
OUT_DIR   = "/Users/doyoung-gil/연구실/데이터/사과"
os.makedirs(OUT_DIR, exist_ok=True)

# 기간(포함)
DATE_COL   = "조사일자(YYYYMMDD)"
DATE_START = pd.Timestamp("2013-01-01")
DATE_END   = pd.Timestamp("2024-12-31")

# ▶ 표준(Core) 스키마: 모델/조인에 공통으로 쓸 컬럼
CORE = [
    "지점ID", "조사년도", "지역-시도", "지역-시군구", "지역-읍면동",
    "상세주소", "좌표-경도", "좌표-위도", "조사회차",
    DATE_COL,
    "작물명", "품종명"
]

# ▶ Extras 보존 여부/방식 (원래 옵션 유지)
KEEP_EXTRAS   = False   # False면 extras 제거
PREFIX_EXTRAS = True    # True면 'CROP__컬럼명' 접두어

# ▶ 추출 대상 병해/해충 + 매칭 키워드(열 이름에 포함되면 "관련 컬럼"으로 채택)
PEST_TARGETS = {
    "복숭아순나방":["복숭아순나방"]
    # "잎집무늬마름병": ["잎집무늬마름병"],
    # "흰잎마름병"    : ["흰잎마름병", "세균성벼흰잎마름병", "벼흰잎마름병"],
    # "깨씨무늬병"    : ["깨씨무늬병"],
    # "이화명나방"    : ["이화명나방"],
    # "벼멸구"        : ["벼멸구"],
    # "흰등멸구"      : ["흰등멸구"],
}

# 탐색 방식
RECURSIVE = False  # 하위 폴더까지 포함하려면 True

# =============================================================
def detect_crop_from_name(fname: str) -> str:
    """파일명에서 작물명 추정. (예: RDA_OBSERVATION_RICE_with_siteid_dt.csv -> RICE)"""
    m = re.search(r"RDA_OBSERVATION_([^_]+)_with_siteid_dt\.csv", fname)
    return m.group(1).upper() if m else "UNKNOWN"

def load_one(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, encoding="utf-8-sig")

    # 작물명 보정(없으면 파일명에서 추정)
    if "작물명" not in df.columns:
        df["작물명"] = detect_crop_from_name(os.path.basename(path))

    # 날짜 파싱 + 기간 필터
    if DATE_COL not in df.columns:
        print(f"[건너뜀] {os.path.basename(path)}: '{DATE_COL}' 컬럼 없음")
        return pd.DataFrame()
    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
    df = df[(df[DATE_COL] >= DATE_START) & (df[DATE_COL] <= DATE_END)]
    if df.empty:
        print(f"[건너뜀] {os.path.basename(path)}: 기간 내 데이터 없음")
        return pd.DataFrame()

    # 조사년도 생성/보정
    if "조사년도" not in df.columns:
        df["조사년도"] = df[DATE_COL].dt.year
    else:
        bad_mask = ~df["조사년도"].astype(str).str.fullmatch(r"\d{4}", na=False)
        df.loc[bad_mask, "조사년도"] = df.loc[bad_mask, DATE_COL].dt.year

    # 정렬
    sort_keys = [c for c in ["지점ID", DATE_COL] if c in df.columns]
    if sort_keys:
        df = df.sort_values(sort_keys)

    return df

def pick_pest_cols(df: pd.DataFrame, pest_keys) -> list:
    """병해충 이름 키워드가 들어간 '모든 관련 컬럼' 선택"""
    cols = []
    for c in df.columns:
        if any(k.casefold() in str(c).casefold() for k in pest_keys):
            cols.append(c)
    return sorted(cols)

# === 메인: 파일 수집 ===
if RECURSIVE:
    files = sorted(glob.glob(os.path.join(INPUT_DIR, "**", "RDA_OBSERVATION_*_with_siteid_dt.csv"), recursive=True))
else:
    files = sorted(glob.glob(os.path.join(INPUT_DIR, "RDA_OBSERVATION_*_with_siteid_dt.csv")))

if not files:
    raise SystemExit(f"대상 파일이 없습니다: {INPUT_DIR}")

# === 로드 & 병합 (병해충별) ===
from collections import defaultdict
pest_to_parts = defaultdict(list)

for fp in files:
    df_part = load_one(fp)
    if df_part.empty:
        continue

    core_present = [c for c in CORE if c in df_part.columns]  # 실제 존재하는 Core만

    for pest_name, pest_keys in PEST_TARGETS.items():
        pest_cols = pick_pest_cols(df_part, pest_keys)
        if not pest_cols:
            continue

        out = df_part[core_present + pest_cols].copy()
        # 병해 관련 값이 전부 결측인 행 제거(원치 않으면 주석 처리)
        out = out.dropna(subset=pest_cols, how="all")
        if out.empty:
            continue

        if KEEP_EXTRAS:
            used = set(core_present + pest_cols)
            extras = [c for c in df_part.columns if c not in used]
            if extras:
                if PREFIX_EXTRAS:
                    crop = out["작물명"].iloc[0] if "작물명" in out.columns and not out.empty \
                           else detect_crop_from_name(os.path.basename(fp))
                    extra_ren = {c: f"{crop}__{c}" for c in extras}
                    out = pd.concat([out, df_part.loc[out.index, extras].rename(columns=extra_ren)], axis=1)
                else:
                    out = pd.concat([out, df_part.loc[out.index, extras]], axis=1)

        pest_to_parts[pest_name].append(out)

if not pest_to_parts:
    raise SystemExit("추출할 데이터가 없습니다. 경로/파일/키워드/기간을 확인하세요.")

# === 저장 (병해충별 → 작물별 파일) ===
for pest_name, parts in pest_to_parts.items():
    all_df = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
    if all_df.empty:
        print(f"[경고] '{pest_name}' 데이터 없음")
        continue

    # 최종 정렬
    sort_keys = [c for c in ["지점ID", DATE_COL] if c in all_df.columns]
    if sort_keys:
        all_df = all_df.sort_values(sort_keys)
    all_df = all_df.drop_duplicates()

    # 작물별로 저장 (원래 코드 스타일 유지)
    if "작물명" in all_df.columns:
        for crop, g in all_df.groupby("작물명", dropna=False):
            # 컬럼 재정렬: Core → pest 관련 → 나머지
            core_present = [c for c in CORE if c in g.columns]
            pest_cols = [c for c in g.columns if c not in core_present and
                         any(k.casefold() in c.casefold() for k in PEST_TARGETS[pest_name])]
            extras = [c for c in g.columns if c not in core_present + pest_cols]
            g = g[core_present + pest_cols + extras]

            save_path = os.path.join(OUT_DIR, f"{pest_name}_{crop}_2013_2024.csv")
            g.to_csv(save_path, index=False, encoding="utf-8-sig")
            print(f"[저장] {save_path} (rows={len(g):,}, cols={len(g.columns)})")
    else:
        # 작물명 컬럼이 없으면 병해충 단일 파일로 저장
        save_path = os.path.join(OUT_DIR, f"{pest_name}_2013_2024.csv")
        all_df.to_csv(save_path, index=False, encoding="utf-8-sig")
        print(f"[저장] {save_path} (rows={len(all_df):,}, cols={len(all_df.columns)})")


[저장] /Users/doyoung-gil/연구실/데이터/사과/복숭아순나방_APPLE_2013_2024.csv (rows=11,278, cols=20)


#### 복숭아순나방

In [ ]:
# -*- coding: utf-8 -*-
import os, re, glob
import pandas as pd

# ================== 설정 ==================
INPUT_DIR = "/Users/doyoung-gil/연구실/데이터/관찰데이터+지점ID+datetime"
OUT_DIR   = "/Users/doyoung-gil/연구실/데이터/사과"
os.makedirs(OUT_DIR, exist_ok=True)

# 기간(포함)
DATE_COL   = "조사일자(YYYYMMDD)"
DATE_START = pd.Timestamp("2013-01-01")
DATE_END   = pd.Timestamp("2024-12-31")

# ▶ 표준(Core) 스키마: 모델/조인에 공통으로 쓸 컬럼
CORE = [
    "지점ID", "조사년도", "지역-시도", "지역-시군구", "지역-읍면동",
    "상세주소", "좌표-경도", "좌표-위도", "조사회차",
    DATE_COL,
    "작물명", "품종명"
]

# ▶ Extras 보존 여부/방식 (원래 옵션 유지)
KEEP_EXTRAS   = False   # False면 extras 제거
PREFIX_EXTRAS = True    # True면 'CROP__컬럼명' 접두어

# ▶ 추출 대상 병해/해충 + 매칭 키워드(열 이름에 포함되면 "관련 컬럼"으로 채택)
# ▶ 추출 대상
PEST_TARGETS = {
    "복숭아순나방": ["복숭아순나방"],
}

import re

def build_pattern(aliases):
    # '한글 글자'에 붙어있지 않을 때만 매치 (붙이 제외)
    parts = [fr"(?<![가-힣]){re.escape(a)}(?![가-힣])" for a in aliases]
    return re.compile("|".join(parts))

# 한번만 컴파일해서 재사용
PEST_PATTERNS = {name: build_pattern(aliases) for name, aliases in PEST_TARGETS.items()}

def pick_pest_cols(df: pd.DataFrame, pest_name: str) -> list:
    pat = PEST_PATTERNS[pest_name]
    return [c for c in df.columns if pat.search(str(c))]

# ↓↓↓ 이 줄은 지워주세요 (df가 아직 없음)
# print(pick_cols(df, "복숭아순나방"))


# 탐색 방식
RECURSIVE = False  # 하위 폴더까지 포함하려면 True

# =============================================================
def detect_crop_from_name(fname: str) -> str:
    """파일명에서 작물명 추정. (예: RDA_OBSERVATION_RICE_with_siteid_dt.csv -> RICE)"""
    m = re.search(r"RDA_OBSERVATION_([^_]+)_with_siteid_dt\.csv", fname)
    return m.group(1).upper() if m else "UNKNOWN"

def load_one(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, encoding="utf-8-sig")

    # 작물명 보정(없으면 파일명에서 추정)
    if "작물명" not in df.columns:
        df["작물명"] = detect_crop_from_name(os.path.basename(path))

    # 날짜 파싱 + 기간 필터
    if DATE_COL not in df.columns:
        print(f"[건너뜀] {os.path.basename(path)}: '{DATE_COL}' 컬럼 없음")
        return pd.DataFrame()
    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
    df = df[(df[DATE_COL] >= DATE_START) & (df[DATE_COL] <= DATE_END)]
    if df.empty:
        print(f"[건너뜀] {os.path.basename(path)}: 기간 내 데이터 없음")
        return pd.DataFrame()

    # 조사년도 생성/보정
    if "조사년도" not in df.columns:
        df["조사년도"] = df[DATE_COL].dt.year
    else:
        bad_mask = ~df["조사년도"].astype(str).str.fullmatch(r"\d{4}", na=False)
        df.loc[bad_mask, "조사년도"] = df.loc[bad_mask, DATE_COL].dt.year

    # 정렬
    sort_keys = [c for c in ["지점ID", DATE_COL] if c in df.columns]
    if sort_keys:
        df = df.sort_values(sort_keys)

    return df


# === 메인: 파일 수집 ===
if RECURSIVE:
    files = sorted(glob.glob(os.path.join(INPUT_DIR, "**", "RDA_OBSERVATION_*_with_siteid_dt.csv"), recursive=True))
else:
    files = sorted(glob.glob(os.path.join(INPUT_DIR, "RDA_OBSERVATION_*_with_siteid_dt.csv")))

if not files:
    raise SystemExit(f"대상 파일이 없습니다: {INPUT_DIR}")

# === 로드 & 병합 (병해충별) ===
from collections import defaultdict
pest_to_parts = defaultdict(list)

for fp in files:
    df_part = load_one(fp)
    if df_part.empty:
        continue

    core_present = [c for c in CORE if c in df_part.columns]  # 실제 존재하는 Core만

    for pest_name in PEST_TARGETS:
        pest_cols = pick_pest_cols(df_part, pest_name)
        if not pest_cols:
            continue

        out = df_part[core_present + pest_cols].copy()
        # 병해 관련 값이 전부 결측인 행 제거(원치 않으면 주석 처리)
        out = out.dropna(subset=pest_cols, how="all")
        if out.empty:
            continue

        if KEEP_EXTRAS:
            used = set(core_present + pest_cols)
            extras = [c for c in df_part.columns if c not in used]
            if extras:
                if PREFIX_EXTRAS:
                    crop = out["작물명"].iloc[0] if "작물명" in out.columns and not out.empty \
                           else detect_crop_from_name(os.path.basename(fp))
                    extra_ren = {c: f"{crop}__{c}" for c in extras}
                    out = pd.concat([out, df_part.loc[out.index, extras].rename(columns=extra_ren)], axis=1)
                else:
                    out = pd.concat([out, df_part.loc[out.index, extras]], axis=1)

        pest_to_parts[pest_name].append(out)

if not pest_to_parts:
    raise SystemExit("추출할 데이터가 없습니다. 경로/파일/키워드/기간을 확인하세요.")

# === 저장 (병해충별 → 작물별 파일) ===
for pest_name, parts in pest_to_parts.items():
    all_df = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
    if all_df.empty:
        print(f"[경고] '{pest_name}' 데이터 없음")
        continue

    # 최종 정렬
    sort_keys = [c for c in ["지점ID", DATE_COL] if c in all_df.columns]
    if sort_keys:
        all_df = all_df.sort_values(sort_keys)
    all_df = all_df.drop_duplicates()

    # 작물별로 저장 (원래 코드 스타일 유지)
    if "작물명" in all_df.columns:
        for crop, g in all_df.groupby("작물명", dropna=False):
            # 컬럼 재정렬: Core → pest 관련 → 나머지
            # --- 저장 루프 내부 재정렬 부분만 수정 ---
            core_present = [c for c in CORE if c in g.columns]
            pat = PEST_PATTERNS[pest_name]
            pest_cols = [c for c in g.columns if c not in core_present and pat.search(str(c))]
            extras = [c for c in g.columns if c not in core_present + pest_cols]
            g = g[core_present + pest_cols + extras]


            save_path = os.path.join(OUT_DIR, f"{pest_name}_{crop}_2013_2024.csv")
            g.to_csv(save_path, index=False, encoding="utf-8-sig")
            print(f"[저장] {save_path} (rows={len(g):,}, cols={len(g.columns)})")
    else:
        # 작물명 컬럼이 없으면 병해충 단일 파일로 저장
        save_path = os.path.join(OUT_DIR, f"{pest_name}_2013_2024.csv")
        all_df.to_csv(save_path, index=False, encoding="utf-8-sig")
        print(f"[저장] {save_path} (rows={len(all_df):,}, cols={len(all_df.columns)})")


[저장] /Users/doyoung-gil/연구실/데이터/사과/복숭아순나방_APPLE_2013_2024.csv (rows=11,274, cols=17)


In [9]:
import pandas as pd
df = pd.read_csv("/Users/doyoung-gil/연구실/데이터/사과/복숭아순나방_APPLE_2013_2024.csv", encoding="utf-8")  # 필요시 euc-kr
print(sorted(df["조사년도"].dropna().unique()))


[2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]


####  # 병해충 데이터의 연도 확인하기

In [8]:
import os, glob
import pandas as pd

OUT_DIR = "/Users/doyoung-gil/연구실/데이터/벼"  # 너가 저장한 폴더
files = sorted(glob.glob(os.path.join(OUT_DIR, "*.csv")))

def read_years_only(path):
    for enc in ("utf-8-sig", "cp949", "euc-kr", "utf-8"):
        try:
            # '조사년도'만 선택해서 읽기 (없으면 빈 DF)
            df = pd.read_csv(path, encoding=enc, usecols=lambda c: c == "조사년도")
            return df
        except Exception:
            continue
    # 마지막 시도
    return pd.read_csv(path, usecols=lambda c: c == "조사년도")

for fp in files:
    df = read_years_only(fp)
    if "조사년도" not in df.columns:
        print(f"[스킵] {os.path.basename(fp)}: '조사년도' 없음")
        continue
    years = pd.to_numeric(df["조사년도"], errors="coerce").dropna().astype(int)
    print(f"{os.path.basename(fp)} → {sorted(years.unique())}")


[스킵] neighbors_master.csv: '조사년도' 없음
[스킵] site_master.csv: '조사년도' 없음
잎집무늬마름병_RICE_1997_2024.csv → [1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
깨씨무늬병_RICE_1997_2024.csv → [2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
벼멸구_RICE_1997_2024.csv → [1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
이화명나방_RICE_1997_2024.csv → [1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
잎도열병_RICE_1997_2024.csv → [1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2

### 기상데이터 + 병해충데이터

### ASOS_AWS 자료에서 DD,GDD 구하기

In [1]:
# add_dd_gdd_to_weather.py
# -*- coding: utf-8 -*-
import os, glob
import numpy as np
import pandas as pd

# ===== 설정 =====
WEATHER_DIR = "/Users/doyoung-gil/Desktop/연구실/데이터/ASOS+AWS"   # ASOS_AWS_YYYY.csv 들이 있는 폴더
PATTERNS    = ["ASOS_AWS_199[7-9].csv", "ASOS_AWS_200[0-9].csv", "ASOS_AWS_2010.csv"] 
BASE_TEMP   = 10.0                                                 # DD/GDD 기준온도(°C)

# ===== 유틸 =====
def pick_series(df: pd.DataFrame, candidates):
    for c in candidates:
        if c in df.columns:
            return pd.to_numeric(df[c], errors="coerce")
    return pd.Series(np.nan, index=df.index, dtype="float64")

def add_dd_gdd(df: pd.DataFrame, base_temp=10.0, site_col="지점"):
    # 날짜
    if "일시" not in df.columns:
        raise ValueError("'일시' 컬럼이 필요합니다.")
    df["일시"] = pd.to_datetime(df["일시"], errors="coerce").dt.normalize()

    # 기온 시리즈(평균기온 우선, 없거나 NaN이면 (최저+최고)/2 대체)
    tmean = pick_series(df, ["평균기온(°C)", "평균기온(℃)", "평균기온"])
    tmin  = pick_series(df, ["최저기온(°C)", "최저기온(℃)", "최저기온"])
    tmax  = pick_series(df, ["최고기온(°C)", "최고기온(℃)", "최고기온"])
    approx = (tmin + tmax) / 2.0
    tmean  = tmean.where(~tmean.isna(), approx)

    # DD, GDD(연도별 리셋)
    df["DD10"] = (tmean - base_temp).clip(lower=0)
    df = df.sort_values([site_col, "일시"]) if site_col in df.columns else df.sort_values("일시")
    df["연도"] = df["일시"].dt.year
    if site_col in df.columns:
        df["GDD10_cum"] = df.groupby([site_col, "연도"], sort=False)["DD10"].cumsum()
    else:
        df["GDD10_cum"] = df.groupby(["연도"], sort=False)["DD10"].cumsum()

    # 보기 좋게 반올림 + 연도 컬럼 제거(원하면 주석 처리)
    df["DD10"] = df["DD10"].round(1)
    df["GDD10_cum"] = df["GDD10_cum"].round(1)
    df.drop(columns=["연도"], inplace=True)

    return df

# ===== 메인 =====
files = sorted({fp
                for pat in PATTERNS
                for fp in glob.glob(os.path.join(WEATHER_DIR, pat))})
if not files:
    pats = [os.path.join(WEATHER_DIR, p) for p in PATTERNS]
    raise SystemExit(f"처리할 파일이 없습니다: {pats}")

for fp in files:
    df = pd.read_csv(fp, encoding="utf-8-sig")
    df_out = add_dd_gdd(df, base_temp=BASE_TEMP, site_col="지점")
    out_path = fp.replace(".csv", "_with_GDD10.csv")
    df_out.to_csv(out_path, index=False, encoding="utf-8-sig", float_format="%.1f")
    print(f"[저장] {out_path}  rows={len(df_out):,}, cols={len(df_out.columns)}")


[저장] /Users/doyoung-gil/Desktop/연구실/데이터/ASOS+AWS/ASOS_AWS_1997_with_GDD10.csv  rows=109,466, cols=13
[저장] /Users/doyoung-gil/Desktop/연구실/데이터/ASOS+AWS/ASOS_AWS_1998_with_GDD10.csv  rows=112,812, cols=13
[저장] /Users/doyoung-gil/Desktop/연구실/데이터/ASOS+AWS/ASOS_AWS_1999_with_GDD10.csv  rows=114,968, cols=13
[저장] /Users/doyoung-gil/Desktop/연구실/데이터/ASOS+AWS/ASOS_AWS_2000_with_GDD10.csv  rows=116,997, cols=13
[저장] /Users/doyoung-gil/Desktop/연구실/데이터/ASOS+AWS/ASOS_AWS_2001_with_GDD10.csv  rows=124,700, cols=13
[저장] /Users/doyoung-gil/Desktop/연구실/데이터/ASOS+AWS/ASOS_AWS_2002_with_GDD10.csv  rows=134,654, cols=13
[저장] /Users/doyoung-gil/Desktop/연구실/데이터/ASOS+AWS/ASOS_AWS_2003_with_GDD10.csv  rows=138,913, cols=13
[저장] /Users/doyoung-gil/Desktop/연구실/데이터/ASOS+AWS/ASOS_AWS_2004_with_GDD10.csv  rows=141,649, cols=13
[저장] /Users/doyoung-gil/Desktop/연구실/데이터/ASOS+AWS/ASOS_AWS_2005_with_GDD10.csv  rows=141,993, cols=13
[저장] /Users/doyoung-gil/Desktop/연구실/데이터/ASOS+AWS/ASOS_AWS_2006_with_GDD10.csv  rows=142,861

### 잎집무늬마름병 관측 장소가 다른병해충 장소를 다 포함하는지 확인

In [12]:
from pathlib import Path
import pandas as pd

OBS_DIR = Path("/Users/doyoung-gil/연구실/데이터/벼")
CROP, SPAN = "RICE", "1997_2024"
NAMES = ["잎도열병","잎집무늬마름병", "흰잎마름병", "깨씨무늬병", "이화명나방", "벼멸구", "흰등멸구"]
BASE_NAME = "잎집무늬마름병"

def read_site_ids(p: Path) -> set[int]:
    # 인코딩 여러 개 시도 + 컬럼명 정규화(공백 제거, 소문자화)
    for enc in ("utf-8-sig", "euc-kr", "cp949", "utf-8"):
        try:
            df = pd.read_csv(p, encoding=enc, low_memory=False)
            break
        except Exception:
            continue
    else:
        df = pd.read_csv(p, low_memory=False)

    df.columns = [c.strip() for c in df.columns]          # 앞뒤 공백 제거
    colmap = {c: c.replace(" ", "") for c in df.columns}  # 중간 공백 제거
    df = df.rename(columns=colmap)

    # 지점ID 대체 이름도 허용
    candidates = [c for c in df.columns if c.lower() in ("지점id", "지점", "siteid", "site_id")]
    if not candidates:
        raise ValueError(f"'지점ID' 컬럼을 찾지 못함: {p.name}")
    col = candidates[0]

    s = pd.to_numeric(df[col], errors="coerce").dropna().astype("int64")
    return set(s.unique().tolist())

def main():
    base_fp = OBS_DIR / f"{BASE_NAME}_{CROP}_{SPAN}.csv"
    if not base_fp.exists():
        raise SystemExit(f"[오류] 기준 파일 없음: {base_fp}")
    base = read_site_ids(base_fp)

    any_missing = False
    rows = []
    for name in NAMES:
        if name == BASE_NAME:
            continue
        fp = OBS_DIR / f"{name}_{CROP}_{SPAN}.csv"
        if not fp.exists():
            print(f"[스킵] 파일 없음: {fp.name}")
            continue
        s = read_site_ids(fp)
        missing = sorted(s - base)
        rows.append([name, len(s), len(missing), len(missing) == 0])
        if missing:
            any_missing = True
            print(f"=== {name} → 기준({BASE_NAME})에 없는 지점 {len(missing)}개 ===")
            print(", ".join(map(str, missing)))
            print()

    # 요약 항상 출력
    if rows:
        summary = pd.DataFrame(rows, columns=["병해충","지점수","누락지점수","완전포함"])
        print("===== 요약 =====")
        print(summary.to_string(index=False))
    if not any_missing:
        print("✅ 모든 병해충 파일의 지점ID가 기준(잎집무늬마름병)에 모두 포함됩니다.")

if __name__ == "__main__":
    main()


===== 요약 =====
  병해충  지점수  누락지점수  완전포함
 잎도열병    0      0  True
흰잎마름병    0      0  True
깨씨무늬병    0      0  True
이화명나방    0      0  True
  벼멸구    0      0  True
 흰등멸구    0      0  True
✅ 모든 병해충 파일의 지점ID가 기준(잎집무늬마름병)에 모두 포함됩니다.


### 잎집마름병 사이트로 지점마스터 파일 만들기

In [13]:
# make_site_master_from_leafblast_fix.py
from pathlib import Path
import pandas as pd

OBS_DIR = Path("/Users/doyoung-gil/연구실/데이터/사과")
BASE    = OBS_DIR / "복숭아순나방_APPLE_2013_2024.csv"
OUT     = OBS_DIR / "site_master.csv"

def read_df(path: Path) -> pd.DataFrame:
    # 지점ID 보존을 위해 dtype=str
    for enc in ("utf-8-sig", "euc-kr", "cp949", "utf-8"):
        try:
            return pd.read_csv(path, encoding=enc, dtype=str, low_memory=False)
        except Exception:
            continue
    return pd.read_csv(path, dtype=str, low_memory=False)

df = read_df(BASE)

# 컬럼 이름 정리
df.columns = [c.strip() for c in df.columns]
df = df.rename(columns={c: c.replace(" ", "") for c in df.columns})

# 필요한 컬럼 찾기
id_col  = next((c for c in df.columns if c.lower() in ("지점id","siteid","site_id","지점")), None)
lat_col = next((c for c in df.columns if "위도" in c), None)
lon_col = next((c for c in df.columns if "경도" in c), None)
if not all([id_col, lat_col, lon_col]):
    raise SystemExit(f"필수 컬럼을 찾지 못했습니다: id={id_col}, lat={lat_col}, lon={lon_col}")

sub = df[[id_col, lat_col, lon_col]].copy()
sub.columns = ["지점ID", "좌표-위도", "좌표-경도"]

# 지점ID는 문자열 유지
sub["지점ID"] = sub["지점ID"].astype(str).str.strip()

# 좌표만 숫자화
sub["좌표-위도"] = pd.to_numeric(sub["좌표-위도"], errors="coerce")
sub["좌표-경도"] = pd.to_numeric(sub["좌표-경도"], errors="coerce")

# 유효값만
sub = sub[(sub["지점ID"] != "") & sub["좌표-위도"].notna() & sub["좌표-경도"].notna()]

# 지점별 대표 좌표(반올림 후 중앙값)
master = (sub.round({"좌표-위도": 6, "좌표-경도": 6})
            .groupby("지점ID", as_index=False)
            .agg({"좌표-위도": "median", "좌표-경도": "median"}))

master.to_csv(OUT, index=False, encoding="utf-8-sig", float_format="%.6f")
print(f"[저장] {OUT} (지점 수={len(master):,})")
print(master.head())


[저장] /Users/doyoung-gil/연구실/데이터/사과/site_master.csv (지점 수=379)
          지점ID      좌표-위도       좌표-경도
0  30523_63004  37.005913  126.552430
1  30551_62994  37.003418  126.561989
2  30636_62943  36.989782  126.590954
3  30644_62932  36.986681  126.593698
4  30670_56490  35.244551  126.622101


### 이웃 관측소 마스터 만들기

In [15]:
# build_neighbors_master_csv.py
# -*- coding: utf-8 -*-
from pathlib import Path
import pandas as pd
import numpy as np

# ===== 설정 =====
EARTH_R   = 6371000.0     # 지구 반경(m)
IDW_POWER = 1.5           # IDW 가중치 지수 (w = 1/d^p)
K_NEIGH   = 3             # 지점ID당 이웃 관측소 개수

WEATHER_DIR = Path("/Users/doyoung-gil/연구실/데이터/ASOS+AWS/GDD")
SITE_MASTER = Path("/Users/doyoung-gil/연구실/데이터/사과/site_master.csv")
OUT_CSV     = Path("/Users/doyoung-gil/연구실/데이터/사과/neighbors_master.csv")
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)

DATE_COL   = "일시"
DATE_START = pd.Timestamp("2013-01-01")
DATE_END   = pd.Timestamp("2025-01-19")


# ===== 함수 =====
def hv(lat1, lon1, lat2, lon2):
    """벡터화 하버사인 거리(m)"""
    lat1 = np.radians(lat1); lon1 = np.radians(lon1)
    lat2 = np.radians(lat2); lon2 = np.radians(lon2)
    dlat = lat2 - lat1; dlon = lon2 - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2.0)**2
    return 2 * EARTH_R * np.arcsin(np.sqrt(a))

def read_weather_stations() -> pd.DataFrame:
    """기상 파일들에서 관측소 좌표만 모아서 고유 목록 반환."""
    parts = []
    for y in range(DATE_START.year, DATE_END.year + 1):
        p = WEATHER_DIR / f"ASOS_AWS_{y}_with_GDD10.csv"
        if not p.exists():
            continue
        w = pd.read_csv(p, encoding="utf-8-sig", low_memory=False)
        # 기간 필터(있을 때만)
        if DATE_COL in w.columns:
            w[DATE_COL] = pd.to_datetime(w[DATE_COL], errors="coerce").dt.normalize()
            w = w[(w[DATE_COL] >= DATE_START) & (w[DATE_COL] <= DATE_END)]
        # 좌표/ID 숫자화
        for c in ["지점", "위도", "경도"]:
            if c in w.columns:
                w[c] = pd.to_numeric(w[c], errors="coerce")
        parts.append(w[["지점", "위도", "경도"]])
    if not parts:
        raise SystemExit("관측소 좌표를 읽을 수 있는 기상 파일이 없습니다.")
    stn = pd.concat(parts, ignore_index=True).dropna().drop_duplicates()
    if stn.empty:
        raise SystemExit("관측소 좌표가 비어 있습니다.")
    return stn

def main():
    # 지점 마스터 로드 (지점ID는 문자열 유지)
    site = pd.read_csv(SITE_MASTER, dtype={"지점ID": str})
    site = site.dropna(subset=["좌표-위도", "좌표-경도"]).copy()
    site["지점ID"] = site["지점ID"].astype(str).str.strip()
    site = site[site["지점ID"] != ""].drop_duplicates(subset=["지점ID"])

    # 관측소 목록 로드
    stn = read_weather_stations()
    stn_lat = stn["위도"].to_numpy()
    stn_lon = stn["경도"].to_numpy()
    stn_ids = stn["지점"].to_numpy()

    rows = []
    for sid, slat, slon in site[["지점ID", "좌표-위도", "좌표-경도"]].itertuples(index=False):
        d = hv(float(slat), float(slon), stn_lat, stn_lon)       # 거리(m)
        k = min(K_NEIGH, len(stn_ids))
        order = np.argsort(d)[:k]
        d_k, ids_k = d[order], stn_ids[order]
        # 가중치(0m 보호)
        if np.isclose(d_k[0], 0.0):
            w = np.zeros_like(d_k); w[0] = 1.0
        else:
            w = 1.0 / np.power(np.maximum(d_k, 1.0), IDW_POWER)
        for r, (stn_id, dist, w_raw) in enumerate(zip(ids_k, d_k, w), start=1):
            rows.append((sid, int(stn_id), r, float(dist), float(w_raw)))

    neigh = pd.DataFrame(rows, columns=["지점ID", "지점", "rank", "거리_m", "w_raw"])
    neigh = neigh.sort_values(["지점ID", "rank"]).drop_duplicates(["지점ID", "rank"])

    # CSV 저장
    neigh.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
    # 간단 요약
    cnt_by_site = neigh.groupby("지점ID")["rank"].max()
    print(f"[저장] {OUT_CSV}  rows={len(neigh):,},  지점수={cnt_by_site.size:,}")
    print("지점별 이웃개수 분포:")
    print(cnt_by_site.value_counts().sort_index())

if __name__ == "__main__":
    main()


[저장] /Users/doyoung-gil/연구실/데이터/사과/neighbors_master.csv  rows=1,137,  지점수=379
지점별 이웃개수 분포:
rank
3    379
Name: count, dtype: int64


### 기상데이터(GDD) + 병해충데이터 병합 -> 시계열테이블

In [5]:
# # join_weather_pest_timeseries_from_base_obs.py
# # -*- coding: utf-8 -*-
# import os, glob
# import numpy as np
# import pandas as pd
# from scipy.spatial import cKDTree

# # ===== 설정 =====
# CROP  = "RICE"  # "APPLE" / "BEAN" / "PEPPER" / "RICE"
# NAMES = ["잎집무늬마름병", "흰잎마름병", "깨씨무늬병", "이화명나방", "벼멸구", "흰등멸구"]

# DATE_START = pd.Timestamp("1997-01-01")
# DATE_END   = pd.Timestamp("2025-01-19")
# SPAN_TAG   = "1997_2024"
# DATE_COL   = "일시"

# # 기상(이미 DD/GDD 포함된 파일들) & 관찰 원천 경로
# WEATHER_DIR = "/Users/doyoung-gil/Desktop/연구실/데이터/ASOS+AWS/GDD"           # ASOS_AWS_YYYY_with_GDD10.csv
# OBS_DIR     = "/Users/doyoung-gil/Desktop/연구실/데이터/벼"                      # {이름}_{CROP}_{SPAN}.csv
# OUT_DIR     = "/Users/doyoung-gil/Desktop/연구실/데이터/벼/벼_joined_samples"    # 결과 저장
# os.makedirs(OUT_DIR, exist_ok=True)

# # 원본 기상 열 → 표준 이름 매핑(입력의 다양한 표기를 표준 열로 통일)
# WX_NAME_MAP = {
#     "일강수량": ["일강수량(mm)", "일강수량"],
#     "최고기온": ["최고기온(°C)", "최고기온(℃)", "최고기온"],
#     "최대 풍속": ["최대풍속(m/s)", "최대풍속", "최대 풍속"],
#     "최저기온": ["최저기온(°C)", "최저기온(℃)", "최저기온"],
#     "평균 풍속": ["평균풍속(m/s)", "평균풍속", "평균 풍속"],
#     "평균기온": ["평균기온(°C)", "평균기온(℃)", "평균기온"],
# }

# # 병해/해충별 키워드(열 이름 매칭용)
# NAME_KEYS = {
#     "잎집무늬마름병": ["잎집무늬마름병"],
#     "흰잎마름병"    : ["흰잎마름병"],
#     "깨씨무늬병"    : ["깨씨무늬병"],
#     "이화명나방"    : ["이화명나방"],
#     "벼멸구"        : ["벼멸구"],
#     "흰등멸구"      : ["흰등멸구"],
# }

# # 관찰데이터에서 항상 보존할 컬럼
# ALWAYS_KEEP = ["조사면적(ha)", "식부면적(ha)"]

# # ===== 유틸 =====
# def load_weather_year(y: int) -> pd.DataFrame:
#     p = os.path.join(WEATHER_DIR, f"ASOS_AWS_{y}_with_GDD10.csv")
#     if not os.path.exists(p):
#         raise FileNotFoundError(f"GDD가 추가된 기상파일이 필요합니다: {p}")
#     w = pd.read_csv(p, encoding="utf-8-sig")
#     w[DATE_COL] = pd.to_datetime(w[DATE_COL], errors="coerce").dt.normalize()
#     for c in ["지점","위도","경도"]:
#         if c in w.columns: w[c] = pd.to_numeric(w[c], errors="coerce")
#     if {"지점", DATE_COL}.issubset(w.columns):
#         w = w.drop_duplicates(subset=["지점", DATE_COL]).sort_values(["지점", DATE_COL])
#     return w

# def load_weather_range(start_ts, end_ts) -> pd.DataFrame:
#     parts = [load_weather_year(y) for y in range(start_ts.year, end_ts.year + 1)]
#     wx = pd.concat(parts, ignore_index=True)
#     wx = wx[(wx[DATE_COL] >= start_ts) & (wx[DATE_COL] <= end_ts)]
#     return wx

# def pick_first(df: pd.DataFrame, candidates: list):
#     for c in candidates:
#         if c in df.columns: return c
#     return None

# def standardize_weather_cols(df: pd.DataFrame) -> pd.DataFrame:
#     out = df.copy()
#     for std, cands in WX_NAME_MAP.items():
#         src = pick_first(out, cands)
#         out[std] = pd.to_numeric(out[src], errors="coerce") if src else np.nan
#     # DD/GDD는 이미 계산되어 있다고 가정
#     if "DD10" not in out.columns and "DD" in out.columns:
#         out.rename(columns={"DD": "DD10"}, inplace=True)
#     if "GDD10_cum" not in out.columns and "GDD" in out.columns:
#         out.rename(columns={"GDD": "GDD10_cum"}, inplace=True)
#     return out

# def pest_related_cols(df: pd.DataFrame, name: str) -> list:
#     keys = NAME_KEYS.get(name, [name])
#     return [c for c in df.columns if any(k.casefold() in str(c).casefold() for k in keys)]

# def pick_rate_col(df: pd.DataFrame, name: str) -> str | None:
#     keys = NAME_KEYS.get(name, [name])
#     cands = [c for c in df.columns if "발생면적률" in str(c) and any(k in str(c) for k in keys)]
#     if not cands: return None
#     def score(col):
#         best = max((len(k) for k in keys if k in col), default=0)
#         return (-best, len(col))
#     return sorted(cands, key=score)[0]

# # ===== 1) 기상(DD/GDD 포함) 로드 & 표준화 =====
# wx_all = load_weather_range(DATE_START, DATE_END)
# wx_all = standardize_weather_cols(wx_all)
# if wx_all.empty:
#     raise SystemExit("기간 내 기상 데이터가 비었습니다.")

# # 최근접 관측소 매핑
# stn = wx_all[["지점", "위도", "경도"]].dropna().drop_duplicates()
# if stn.empty:
#     raise SystemExit("기상 관측소 좌표가 없습니다.")
# tree = cKDTree(stn[["위도", "경도"]].to_numpy())

# # ===== 2) 병해/해충별 병합(시계열) =====
# for name in NAMES:
#     # 관찰 파일 우선 경로
#     expect = os.path.join(OBS_DIR, f"{name}_{CROP}_{SPAN_TAG}.csv")
#     obs_path = expect if os.path.exists(expect) else None
#     if not obs_path:
#         hits = glob.glob(os.path.join(OBS_DIR, f"*{name}*{CROP}*{SPAN_TAG}*.csv"))
#         obs_path = sorted(hits)[0] if hits else None
#     if not obs_path:
#         print(f"[건너뜀] 관찰 파일 없음: {name}_{CROP}_{SPAN_TAG}.csv")
#         continue

#     obs = pd.read_csv(obs_path, encoding="utf-8-sig")

#     # 필수 컬럼 점검/보정
#     need = {"지점ID","좌표-위도","좌표-경도"}
#     if not need.issubset(obs.columns):
#         print(f"[건너뜀] '{name}': 지점 좌표/ID 누락 → {need - set(obs.columns)}")
#         continue

#     if DATE_COL not in obs.columns and "조사일자(YYYYMMDD)" in obs.columns:
#         obs[DATE_COL] = pd.to_datetime(obs["조사일자(YYYYMMDD)"], errors="coerce").dt.normalize()
#     else:
#         obs[DATE_COL] = pd.to_datetime(obs.get(DATE_COL), errors="coerce").dt.normalize()

#     # 기간 필터
#     obs = obs[(obs[DATE_COL] >= DATE_START) & (obs[DATE_COL] <= DATE_END)].copy()
#     if obs.empty:
#         print(f"[건너뜀] '{name}': 기간 내 관찰 없음")
#         continue

#     # 지점ID → 최근접 관측소 '지점' 매핑
#     site_xy = obs[["지점ID", "좌표-위도", "좌표-경도"]].dropna().drop_duplicates()
#     dist, idx = tree.query(site_xy[["좌표-위도", "좌표-경도"]].to_numpy(), k=1)
#     site_map = site_xy.assign(지점=stn.iloc[idx]["지점"].to_numpy())[["지점ID", "지점"]]

#     # 기준 그리드: 모든 지점ID × 날짜
#     days = pd.DataFrame({DATE_COL: pd.date_range(DATE_START, DATE_END, freq="D")})
#     base = site_map.assign(key=1).merge(days.assign(key=1), on="key").drop(columns="key")

#     # --- 기상 LEFT 조인 (표준명 + DD/GDD 사용)
#     wx_use = wx_all[["지점", DATE_COL, "일강수량", "최고기온", "최대 풍속", "최저기온", "평균 풍속", "평균기온", "DD10", "GDD10_cum"]]
#     out = base.merge(wx_use, on=["지점", DATE_COL], how="left")

#     # --- 관찰 LEFT 조인: 병해 관련 전 컬럼 + 조사/식부면적 보존
#     pest_cols = pest_related_cols(obs, name)
#     keep_extra = [c for c in ALWAYS_KEEP if c in obs.columns]
#     obs_keep_cols = ["지점ID", DATE_COL] + pest_cols + keep_extra

#     # 같은 날 같은 지점 중복 관찰 정리: 조사면적=합산, 식부면적=first, 나머지는 first
#     agg = {}
#     if "조사면적(ha)" in obs_keep_cols: agg["조사면적(ha)"] = "sum"
#     if "식부면적(ha)" in obs_keep_cols: agg["식부면적(ha)"] = "first"
#     for c in pest_cols:
#         if c not in agg: agg[c] = "first"

#     obs_daily = (obs[obs_keep_cols]
#                  .groupby(["지점ID", DATE_COL], as_index=False)
#                  .agg(agg) if agg else obs[obs_keep_cols])

#     out = out.merge(obs_daily, on=["지점ID", DATE_COL], how="left")

#     # 관측값/발생여부: 발생면적률 사용(있을 때만)
#     rate_col = pick_rate_col(out, name)
#     if rate_col:
#         out["관측값"] = pd.to_numeric(out[rate_col], errors="coerce")
#         out["발생여부"] = (out["관측값"].fillna(0) > 0).astype(int)
#     else:
#         out["관측값"] = np.nan
#         out["발생여부"] = np.nan
#         print(f"[알림] '{name}': '발생면적률' 컬럼을 찾지 못해 관측값/발생여부는 NaN")

#     # === 최종 출력 컬럼명(단위 포함) 구성 ===
#     # 표준 컬럼 → 단위 있는 이름 복제
#     out["일강수량(mm)"]   = out["일강수량"]
#     out["최고기온(°C)"]   = out["최고기온"]
#     out["최대 풍속(m/s)"] = out["최대 풍속"]
#     out["최저기온(°C)"]   = out["최저기온"]
#     out["평균 풍속(m/s)"] = out["평균 풍속"]
#     out["평균기온(°C)"]   = out["평균기온"]

#     # ===== 컬럼 순서(요청 고정) =====
#     front = ["지점ID", "지점", DATE_COL, "조사면적(ha)", "식부면적(ha)",
#              "일강수량(mm)", "최고기온(°C)", "최대 풍속(m/s)", "최저기온(°C)", "평균 풍속(m/s)", "평균기온(°C)"]

#     # 병해 관련 전 컬럼(중복/앞부분 제외, 조사/식부면적 제외)
#     pest_cols = [c for c in pest_cols if c not in set(front + ALWAYS_KEEP)]
#     tail  = ["DD10", "GDD10_cum", "관측값", "발생여부"]
#     cols  = front + pest_cols + tail

#     # 누락 컬럼 채우기(NaN 생성) 후 선택
#     for c in cols:
#         if c not in out.columns:
#             out[c] = np.nan

#     out = (out.loc[:, cols]
#               .drop_duplicates(subset=["지점ID", DATE_COL])
#               .sort_values(["지점ID", DATE_COL])
#               .reset_index(drop=True))

#     # 저장
#     save_path = os.path.join(OUT_DIR, f"joined_SAMPLE_GDD_{SPAN_TAG}_{CROP}_{name}.csv")
#     out.to_csv(save_path, index=False, encoding="utf-8-sig", float_format="%.1f")
#     print(f"[저장] {save_path}  rows={len(out):,}, cols={len(out.columns)}")


[저장] /Users/doyoung-gil/Desktop/연구실/데이터/벼/벼_joined_samples/joined_SAMPLE_GDD_1997_2024_RICE_잎집무늬마름병.csv  rows=28,391,666, cols=23
[저장] /Users/doyoung-gil/Desktop/연구실/데이터/벼/벼_joined_samples/joined_SAMPLE_GDD_1997_2024_RICE_흰잎마름병.csv  rows=28,340,436, cols=23
[저장] /Users/doyoung-gil/Desktop/연구실/데이터/벼/벼_joined_samples/joined_SAMPLE_GDD_1997_2024_RICE_깨씨무늬병.csv  rows=28,330,190, cols=23
[저장] /Users/doyoung-gil/Desktop/연구실/데이터/벼/벼_joined_samples/joined_SAMPLE_GDD_1997_2024_RICE_이화명나방.csv  rows=28,309,698, cols=23
[저장] /Users/doyoung-gil/Desktop/연구실/데이터/벼/벼_joined_samples/joined_SAMPLE_GDD_1997_2024_RICE_벼멸구.csv  rows=28,330,190, cols=23
[저장] /Users/doyoung-gil/Desktop/연구실/데이터/벼/벼_joined_samples/joined_SAMPLE_GDD_1997_2024_RICE_흰등멸구.csv  rows=28,360,928, cols=23


In [ ]:
# join_with_neighbors_master.py
# -*- coding: utf-8 -*-
import os, glob
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

# ======================= 설정 =======================
CROP  = "APPLE"  # "APPLE" / "BEAN" / "PEPPER" / "RICE"
NAMES = ["복숭아순나방"]
# NAMES = ["잎집무늬마름병", "흰잎마름병", "깨씨무늬병", "이화명나방", "벼멸구", "흰등멸구"]

DATE_START = pd.Timestamp("2013-01-01")
DATE_END   = pd.Timestamp("2025-01-31")
SPAN_TAG   = "2013_2024"
DATE_COL   = "일시"

# 경로
WEATHER_DIR = "/Users/doyoung-gil/연구실/데이터/ASOS+AWS/GDD"             # ASOS_AWS_YYYY_with_GDD10.csv
OBS_DIR     = "/Users/doyoung-gil/연구실/데이터/사과"                        # {이름}_{CROP}_{SPAN}.csv
NEIGH_CSV   = "/Users/doyoung-gil/연구실/데이터/사과/neighbors_master.csv"
OUT_DIR     = "/Users/doyoung-gil/연구실/데이터/사과/사과_joined_samples"      # 결과 저장
os.makedirs(OUT_DIR, exist_ok=True)

# ===== 결측 보강 설정 =====
FILL_METHOD = "nearest"   # "nearest" or "idw"
IDW_POWER   = 1.5         # idw일 때만 사용

# 원본 기상 열 → 표준 이름 매핑
WX_NAME_MAP = {
    "일강수량": ["일강수량(mm)", "일강수량"],
    "최고기온": ["최고기온(°C)", "최고기온(℃)", "최고기온"],
    "최저기온": ["최저기온(°C)", "최저기온(℃)", "최저기온"],
    "평균기온": ["평균기온(°C)", "평균기온(℃)", "평균기온"],
    "최대 풍속": ["최대 풍속(m/s)", "최대풍속(m/s)", "최대 풍속", "최대풍속",
               "최대 순간 풍속(m/s)", "최대순간풍속(m/s)"],   # ← 두 개 추가
    "평균 풍속": ["평균 풍속(m/s)", "평균풍속(m/s)", "평균 풍속", "평균풍속"],
}


# 병해/해충별 키워드(열 이름 매칭용)
NAME_KEYS = {
    "복숭아순나방": ["복숭아순나방"]
    # "잎집무늬마름병": ["잎집무늬마름병"],
    # "흰잎마름병"    : ["흰잎마름병"],
    # "깨씨무늬병"    : ["깨씨무늬병"],
    # "이화명나방"    : ["이화명나방"],
    # "벼멸구"        : ["벼멸구"],
    # "흰등멸구"      : ["흰등멸구"],
}

ALWAYS_KEEP = ["조사면적(ha)", "식부면적(ha)"]

# ======================= 유틸 =======================
def load_weather_year(y: int) -> pd.DataFrame | None:
    p = os.path.join(WEATHER_DIR, f"ASOS_AWS_{y}_with_GDD10.csv")
    if not os.path.exists(p):
        return None
    w = pd.read_csv(p, encoding="utf-8-sig", low_memory=False)
    w[DATE_COL] = pd.to_datetime(w[DATE_COL], errors="coerce").dt.normalize()
    for c in ["지점","위도","경도"]:
        if c in w.columns: w[c] = pd.to_numeric(w[c], errors="coerce")
    if {"지점", DATE_COL}.issubset(w.columns):
        w = w.drop_duplicates(subset=["지점", DATE_COL]).sort_values(["지점", DATE_COL])
    return w

def load_weather_range(start_ts, end_ts) -> pd.DataFrame:
    parts = []
    for y in range(start_ts.year, end_ts.year + 1):
        df = load_weather_year(y)
        if df is not None and not df.empty:
            parts.append(df)
    if not parts:
        raise FileNotFoundError("기간 내 기상 파일을 찾지 못했습니다.")
    wx = pd.concat(parts, ignore_index=True)
    wx = wx[(wx[DATE_COL] >= start_ts) & (wx[DATE_COL] <= end_ts)]
    return wx

def pick_first(df: pd.DataFrame, candidates: list):
    for c in candidates:
        if c in df.columns: return c
    return None

def standardize_weather_cols(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for std, cands in WX_NAME_MAP.items():
        src = pick_first(out, cands)
        out[std] = pd.to_numeric(out[src], errors="coerce") if src else np.nan
    if "DD10" not in out.columns and "DD" in out.columns:
        out.rename(columns={"DD": "DD10"}, inplace=True)
    if "GDD10_cum" not in out.columns and "GDD" in out.columns:
        out.rename(columns={"GDD": "GDD10_cum"}, inplace=True)
    return out

def pest_related_cols(df: pd.DataFrame, name: str) -> list:
    keys = [k.casefold() for k in NAME_KEYS.get(name, [name])]
    cols = []
    for c in df.columns:
        cl = str(c).casefold()
        if any(k in cl for k in keys):
            cols.append(c)
    return cols

def pick_rate_col(df: pd.DataFrame, name: str) -> str | None:
    keys = [k.casefold() for k in NAME_KEYS.get(name, [name])]
    cols_lower = {c: str(c).casefold() for c in df.columns}
    cands = [c for c, cl in cols_lower.items() if ("발생면적률" in cl) and any(k in cl for k in keys)]
    if not cands:
        return None
    def score(col):
        cl = cols_lower[col]
        best = max((len(k) for k in keys if k in cl), default=0)
        return (-best, len(col))
    return sorted(cands, key=score)[0]

def aggregate_weather_by_neighbors(wx_all: pd.DataFrame,
                                   neighbors: pd.DataFrame,
                                   date_col: str,
                                   agg_cols: list,
                                   method: str = "idw",
                                   idw_power: float = 1.5) -> pd.DataFrame:
    """
    neighbors: ['지점ID','지점','rank','거리_m','w_raw'] (지점ID는 문자열)
    반환: ['지점ID', date_col] + agg_cols  (전 기간 매일)
    """
    # 사이트/스테이션 인덱싱
    site_ids = neighbors['지점ID'].astype(str).drop_duplicates().to_list()
    stn_ids  = wx_all['지점'].drop_duplicates().to_list()
    site_idx = {sid: i for i, sid in enumerate(site_ids)}
    stn_idx  = {sid: j for j, sid in enumerate(stn_ids)}
    n_sites  = len(site_ids)
    dates    = pd.date_range(DATE_START, DATE_END, freq="D")
    n_days   = len(dates)

    # 희소 가중치
    r_idx = neighbors['지점ID'].map(site_idx).to_numpy()
    c_idx = neighbors['지점'].map(stn_idx).to_numpy()
    w_raw = neighbors['w_raw'].astype(float).to_numpy()
    W = csr_matrix((w_raw, (r_idx, c_idx)), shape=(n_sites, len(stn_ids)))

    out = pd.DataFrame({
        '지점ID': np.repeat(site_ids, n_days),
        date_col: np.tile(dates.values, n_sites)
    })

    for col in agg_cols:
        mat = (wx_all.pivot(index=date_col, columns='지점', values=col)
                        .reindex(index=dates, columns=stn_ids))
        X = mat.to_numpy(dtype=float)          # (days x stations)
        valid = np.isfinite(X).astype(float)
        X_filled = np.nan_to_num(X, nan=0.0)

        if method == "idw":
            N = W.dot(X_filled.T).astype(float)    # (sites x days)
            D = W.dot(valid.T).astype(float)
            R = np.divide(N, D, out=np.full_like(N, np.nan), where=(D > 0))
            out[col] = R.reshape(n_sites * n_days, order='C')

        elif method == "nearest":
            # rank별 스테이션 인덱스 테이블 (sites x K)
            piv = (neighbors.sort_values(['지점ID', 'rank'])
                             .drop_duplicates(['지점ID','rank'])
                             .pivot(index='지점ID', columns='rank', values='지점')
                             .reindex(index=site_ids))
            idx_by_rank = piv.applymap(stn_idx.get).to_numpy(dtype=float)  # NaN 가능
            K = idx_by_rank.shape[1]
            R = np.full((n_days, n_sites), np.nan, dtype=float)
            for r in range(K):
                idx_r = idx_by_rank[:, r]
                mask_valid = np.isfinite(idx_r)
                if not mask_valid.any(): continue
                cols_r = idx_r.copy(); cols_r[~mask_valid] = 0
                vals_r = X[:, cols_r.astype(int)]
                vals_r[:, ~mask_valid] = np.nan
                fill_mask = np.isnan(R) & np.isfinite(vals_r)
                R[fill_mask] = vals_r[fill_mask]
            out[col] = R.T.reshape(n_sites * n_days, order='C')
        else:
            raise ValueError("method must be 'idw' or 'nearest'")
    return out

# ======================= 메인 =======================
def main():
    # 1) 기상 로드 & 표준화
    wx_all = load_weather_range(DATE_START, DATE_END)
    wx_all = standardize_weather_cols(wx_all)
    stn = wx_all[["지점", "위도", "경도"]].dropna().drop_duplicates()
    if stn.empty:
        raise SystemExit("기상 관측소 좌표가 없습니다.")

    # 2) 이웃 마스터 로드 (CSV)
    neighbors_master = pd.read_csv(NEIGH_CSV, dtype={"지점ID": str})
    neighbors_master = (neighbors_master.sort_values(['지점ID','rank'])
                                         .drop_duplicates(['지점ID','rank']))

    # 3) 병해/해충별 처리
    for name in NAMES:
        fp = os.path.join(OBS_DIR, f"{name}_{CROP}_{SPAN_TAG}.csv")
        if not os.path.exists(fp):
            print(f"[스킵] 관찰 파일 없음: {fp}")
            continue

        obs = pd.read_csv(fp, encoding="utf-8-sig", low_memory=False)
        # 날짜 보정
        if DATE_COL not in obs.columns and "조사일자(YYYYMMDD)" in obs.columns:
            obs[DATE_COL] = pd.to_datetime(obs["조사일자(YYYYMMDD)"], errors="coerce").dt.normalize()
        else:
            obs[DATE_COL] = pd.to_datetime(obs.get(DATE_COL), errors="coerce").dt.normalize()
        obs = obs[(obs[DATE_COL] >= DATE_START) & (obs[DATE_COL] <= DATE_END)].copy()
        if obs.empty:
            print(f"[스킵] '{name}': 기간 내 관찰 없음")
            continue

        # 이번 병해충에 등장하는 지점ID만 슬라이스
        site_ids = pd.Series(obs["지점ID"].astype(str).dropna().unique())
        neighbors = neighbors_master[neighbors_master["지점ID"].isin(site_ids)]
        if neighbors.empty:
            print(f"[스킵] '{name}': neighbors가 비어 있음(지점ID 매칭 실패)")
            continue

        # 전 기간 base (모든 지점ID × 날짜)
        days = pd.date_range(DATE_START, DATE_END, freq="D")
        base = (pd.DataFrame({"지점ID": site_ids})
                  .assign(key=1)
                  .merge(pd.DataFrame({DATE_COL: days, "key":1}), on="key")
                  .drop(columns="key"))

        agg_cols = ["일강수량", "최고기온", "최대 풍속", "최저기온", "평균 풍속", "평균기온", "DD10", "GDD10_cum"]
        wx_use = wx_all[["지점", DATE_COL] + agg_cols].copy()

        # 결측 보강 기상 계산
        wx_filled = aggregate_weather_by_neighbors(wx_use, neighbors, DATE_COL, agg_cols,
                                                   method=FILL_METHOD, idw_power=IDW_POWER)
        out = base.merge(wx_filled, on=["지점ID", DATE_COL], how="left")

        # 관찰 조인
        pest_cols = pest_related_cols(obs, name)
        keep_extra = [c for c in ALWAYS_KEEP if c in obs.columns]
        obs_keep_cols = ["지점ID", DATE_COL] + pest_cols + keep_extra

        agg = {}
        if "조사면적(ha)" in obs_keep_cols: agg["조사면적(ha)"] = "sum"
        if "식부면적(ha)" in obs_keep_cols: agg["식부면적(ha)"] = "first"
        for c in pest_cols:
            if c not in agg: agg[c] = "first"
        obs_daily = (obs[obs_keep_cols]
                     .groupby(["지점ID", DATE_COL], as_index=False)
                     .agg(agg) if agg else obs[obs_keep_cols])

        out = out.merge(obs_daily, on=["지점ID", DATE_COL], how="left")

        # 관측값/발생여부
        rate_col = pick_rate_col(out, name)
        if rate_col:
            out["관측값"] = pd.to_numeric(out[rate_col], errors="coerce")
            out["발생여부"] = (out["관측값"].fillna(0) > 0).astype(int)
        else:
            out["관측값"] = np.nan
            out["발생여부"] = np.nan
            print(f"[알림] '{name}': '발생면적률' 컬럼을 찾지 못함 → 관측값/발생여부 NaN")

        # 단위 포함 컬럼 복제
        out["일강수량(mm)"]   = out["일강수량"]
        out["최고기온(°C)"]   = out["최고기온"]
        out["최대 풍속(m/s)"] = out["최대 풍속"]
        out["최저기온(°C)"]   = out["최저기온"]
        out["평균 풍속(m/s)"] = out["평균 풍속"]
        out["평균기온(°C)"]   = out["평균기온"]

        # 컬럼 순서
        front = ["지점ID", DATE_COL, "조사면적(ha)", "식부면적(ha)",
                 "일강수량(mm)", "최고기온(°C)", "최대 풍속(m/s)",
                 "최저기온(°C)", "평균 풍속(m/s)", "평균기온(°C)"]
        pest_cols = [c for c in pest_cols if c not in set(front + ALWAYS_KEEP)]
        tail  = ["DD10", "GDD10_cum", "관측값", "발생여부"]
        cols  = front + pest_cols + tail
        for c in cols:
            if c not in out.columns: out[c] = np.nan

        out = (out.loc[:, cols]
                 .drop_duplicates(subset=["지점ID", DATE_COL])
                 .sort_values(["지점ID", DATE_COL])
                 .reset_index(drop=True))

        save_path = os.path.join(OUT_DIR, f"joined_SAMPLE_GDD_{SPAN_TAG}_{CROP}_{name}.csv")
        out.to_csv(save_path, index=False, encoding="utf-8-sig", float_format="%.1f")
        print(f"[저장] {save_path}  rows={len(out):,}, cols={len(out.columns)}")

if __name__ == "__main__":
    main()


/var/folders/2j/__kxcrvx5ss949vx7sspc6th0000gn/T/ipykernel_48329/2216158541.py:165: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  idx_by_rank = piv.applymap(stn_idx.get).to_numpy(dtype=float)  # NaN 가능
/var/folders/2j/__kxcrvx5ss949vx7sspc6th0000gn/T/ipykernel_48329/2216158541.py:165: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  idx_by_rank = piv.applymap(stn_idx.get).to_numpy(dtype=float)  # NaN 가능
/var/folders/2j/__kxcrvx5ss949vx7sspc6th0000gn/T/ipykernel_48329/2216158541.py:165: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  idx_by_rank = piv.applymap(stn_idx.get).to_numpy(dtype=float)  # NaN 가능
/var/folders/2j/__kxcrvx5ss949vx7sspc6th0000gn/T/ipykernel_48329/2216158541.py:165: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  idx_by_rank = piv.applymap(stn_idx.get).to_numpy(dtype=float)  # NaN 가능
/var/folders/2j/__kxcrvx5ss949vx7sspc6th

[저장] /Users/doyoung-gil/연구실/데이터/벼/벼_joined_samples/joined_SAMPLE_GDD_1997_2024_RICE_잎도열병.csv  rows=28,404,402, cols=22


#### 복숭아순나방

In [17]:
# join_with_neighbors_master.py
# -*- coding: utf-8 -*-
import os, glob, re
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

# ======================= 설정 =======================
CROP  = "APPLE"  # "APPLE" / "BEAN" / "PEPPER" / "RICE"
NAMES = ["복숭아순나방"]
# NAMES = ["잎집무늬마름병", "흰잎마름병", "깨씨무늬병", "이화명나방", "벼멸구", "흰등멸구"]

DATE_START = pd.Timestamp("2013-01-01")
DATE_END   = pd.Timestamp("2025-01-31")
SPAN_TAG   = "2013_2024"
DATE_COL   = "일시"

# 경로
WEATHER_DIR = "/Users/doyoung-gil/연구실/데이터/ASOS+AWS/GDD"             # ASOS_AWS_YYYY_with_GDD10.csv
OBS_DIR     = "/Users/doyoung-gil/연구실/데이터/사과"                        # {이름}_{CROP}_{SPAN}.csv
NEIGH_CSV   = "/Users/doyoung-gil/연구실/데이터/사과/neighbors_master.csv"
OUT_DIR     = "/Users/doyoung-gil/연구실/데이터/사과/사과_joined_samples"      # 결과 저장
os.makedirs(OUT_DIR, exist_ok=True)

# ===== 결측 보강 설정 =====
FILL_METHOD = "nearest"   # "nearest" or "idw"
IDW_POWER   = 1.5         # idw일 때만 사용

# 원본 기상 열 → 표준 이름 매핑
WX_NAME_MAP = {
    "일강수량": ["일강수량(mm)", "일강수량"],
    "최고기온": ["최고기온(°C)", "최고기온(℃)", "최고기온"],
    "최저기온": ["최저기온(°C)", "최저기온(℃)", "최저기온"],
    "평균기온": ["평균기온(°C)", "평균기온(℃)", "평균기온"],
    "최대 풍속": ["최대 풍속(m/s)", "최대풍속(m/s)", "최대 풍속", "최대풍속",
               "최대 순간 풍속(m/s)", "최대순간풍속(m/s)"],
    "평균 풍속": ["평균 풍속(m/s)", "평균풍속(m/s)", "평균 풍속", "평균풍속"],
}

# 병해/해충별 키워드(열 이름 매칭용)
NAME_KEYS = {
    "복숭아순나방": ["복숭아순나방"]
}

ALWAYS_KEEP = ["품종명"]

# === 관측값 우선 컬럼 매핑 (여기만 추가) ===
PREFERRED_OBS_COLS = {
    "복숭아순나방": ["(트랩)복숭아순나방-마리수"]
}

# ======================= 유틸 =======================
def load_weather_year(y: int) -> pd.DataFrame | None:
    p = os.path.join(WEATHER_DIR, f"ASOS_AWS_{y}_with_GDD10.csv")
    if not os.path.exists(p):
        return None
    w = pd.read_csv(p, encoding="utf-8-sig", low_memory=False)
    w[DATE_COL] = pd.to_datetime(w[DATE_COL], errors="coerce").dt.normalize()
    for c in ["지점","위도","경도"]:
        if c in w.columns: w[c] = pd.to_numeric(w[c], errors="coerce")
    if {"지점", DATE_COL}.issubset(w.columns):
        w = w.drop_duplicates(subset=["지점", DATE_COL]).sort_values(["지점", DATE_COL])
    return w

def load_weather_range(start_ts, end_ts) -> pd.DataFrame:
    parts = []
    for y in range(start_ts.year, end_ts.year + 1):
        df = load_weather_year(y)
        if df is not None and not df.empty:
            parts.append(df)
    if not parts:
        raise FileNotFoundError("기간 내 기상 파일을 찾지 못했습니다.")
    wx = pd.concat(parts, ignore_index=True)
    wx = wx[(wx[DATE_COL] >= start_ts) & (wx[DATE_COL] <= end_ts)]
    return wx

def pick_first(df: pd.DataFrame, candidates: list):
    for c in candidates:
        if c in df.columns: return c
    return None

def standardize_weather_cols(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for std, cands in WX_NAME_MAP.items():
        src = pick_first(out, cands)
        out[std] = pd.to_numeric(out[src], errors="coerce") if src else np.nan
    if "DD10" not in out.columns and "DD" in out.columns:
        out.rename(columns={"DD": "DD10"}, inplace=True)
    if "GDD10_cum" not in out.columns and "GDD" in out.columns:
        out.rename(columns={"GDD": "GDD10_cum"}, inplace=True)
    return out

def pest_related_cols(df: pd.DataFrame, name: str) -> list:
    keys = [k.casefold() for k in NAME_KEYS.get(name, [name])]
    cols = []
    for c in df.columns:
        cl = str(c).casefold()
        if any(k in cl for k in keys):
            cols.append(c)
    return cols

def pick_rate_col(df: pd.DataFrame, name: str) -> str | None:
    keys = [k.casefold() for k in NAME_KEYS.get(name, [name])]
    cols_lower = {c: str(c).casefold() for c in df.columns}
    cands = [c for c, cl in cols_lower.items() if ("발생면적률" in cl) and any(k in cl for k in keys)]
    if not cands:
        return None
    def score(col):
        cl = cols_lower[col]
        best = max((len(k) for k in keys if k in cl), default=0)
        return (-best, len(col))
    return sorted(cands, key=score)[0]

# === 관측값 컬럼 선택 함수 (여기만 추가) ===
def build_ko_word_pat(term: str):
    # 한글 경계: '복숭아순나방붙이' 같은 붙어있는 단어는 제외
    return re.compile(fr"(?<![가-힣]){re.escape(term)}(?![가-힣])", flags=re.IGNORECASE)

def pick_obs_value_col(df: pd.DataFrame, name: str) -> str | None:
    """
    1순위: PREFERRED_OBS_COLS[name] 중 존재하는 첫 컬럼(정확 매치 → 경계기반 부분매치 순)
    2순위: 없으면 '발생면적률' 폴백
    """
    prefs = PREFERRED_OBS_COLS.get(name, [])
    for p in prefs:
        if p in df.columns:
            return p
        pat = build_ko_word_pat(p)
        hits = [c for c in df.columns if pat.search(str(c))]
        if hits:
            return sorted(hits, key=len)[0]
    return pick_rate_col(df, name)

def aggregate_weather_by_neighbors(wx_all: pd.DataFrame,
                                   neighbors: pd.DataFrame,
                                   date_col: str,
                                   agg_cols: list,
                                   method: str = "idw",
                                   idw_power: float = 1.5) -> pd.DataFrame:
    """
    neighbors: ['지점ID','지점','rank','거리_m','w_raw'] (지점ID는 문자열)
    반환: ['지점ID', date_col] + agg_cols  (전 기간 매일)
    """
    # 사이트/스테이션 인덱싱
    site_ids = neighbors['지점ID'].astype(str).drop_duplicates().to_list()
    stn_ids  = wx_all['지점'].drop_duplicates().to_list()
    site_idx = {sid: i for i, sid in enumerate(site_ids)}
    stn_idx  = {sid: j for j, sid in enumerate(stn_ids)}
    n_sites  = len(site_ids)
    dates    = pd.date_range(DATE_START, DATE_END, freq="D")
    n_days   = len(dates)

    # 희소 가중치
    r_idx = neighbors['지점ID'].map(site_idx).to_numpy()
    c_idx = neighbors['지점'].map(stn_idx).to_numpy()
    w_raw = neighbors['w_raw'].astype(float).to_numpy()
    W = csr_matrix((w_raw, (r_idx, c_idx)), shape=(n_sites, len(stn_ids)))

    out = pd.DataFrame({
        '지점ID': np.repeat(site_ids, n_days),
        date_col: np.tile(dates.values, n_sites)
    })

    for col in agg_cols:
        mat = (wx_all.pivot(index=date_col, columns='지점', values=col)
                        .reindex(index=dates, columns=stn_ids))
        X = mat.to_numpy(dtype=float)          # (days x stations)
        valid = np.isfinite(X).astype(float)
        X_filled = np.nan_to_num(X, nan=0.0)

        if method == "idw":
            N = W.dot(X_filled.T).astype(float)    # (sites x days)
            D = W.dot(valid.T).astype(float)
            R = np.divide(N, D, out=np.full_like(N, np.nan), where=(D > 0))
            out[col] = R.reshape(n_sites * n_days, order='C')

        elif method == "nearest":
            # rank별 스테이션 인덱스 테이블 (sites x K)
            piv = (neighbors.sort_values(['지점ID', 'rank'])
                             .drop_duplicates(['지점ID','rank'])
                             .pivot(index='지점ID', columns='rank', values='지점')
                             .reindex(index=site_ids))
            idx_by_rank = piv.applymap(stn_idx.get).to_numpy(dtype=float)  # NaN 가능
            K = idx_by_rank.shape[1]
            R = np.full((n_days, n_sites), np.nan, dtype=float)
            for r in range(K):
                idx_r = idx_by_rank[:, r]
                mask_valid = np.isfinite(idx_r)
                if not mask_valid.any(): continue
                cols_r = idx_r.copy(); cols_r[~mask_valid] = 0
                vals_r = X[:, cols_r.astype(int)]
                vals_r[:, ~mask_valid] = np.nan
                fill_mask = np.isnan(R) & np.isfinite(vals_r)
                R[fill_mask] = vals_r[fill_mask]
            out[col] = R.T.reshape(n_sites * n_days, order='C')
        else:
            raise ValueError("method must be 'idw' or 'nearest'")
    return out

# ======================= 메인 =======================
def main():
    # 1) 기상 로드 & 표준화
    wx_all = load_weather_range(DATE_START, DATE_END)
    wx_all = standardize_weather_cols(wx_all)
    stn = wx_all[["지점", "위도", "경도"]].dropna().drop_duplicates()
    if stn.empty:
        raise SystemExit("기상 관측소 좌표가 없습니다.")

    # 2) 이웃 마스터 로드 (CSV)
    neighbors_master = pd.read_csv(NEIGH_CSV, dtype={"지점ID": str})
    neighbors_master = (neighbors_master.sort_values(['지점ID','rank'])
                                         .drop_duplicates(['지점ID','rank']))

    # 3) 병해/해충별 처리
    for name in NAMES:
        fp = os.path.join(OBS_DIR, f"{name}_{CROP}_{SPAN_TAG}.csv")
        if not os.path.exists(fp):
            print(f"[스킵] 관찰 파일 없음: {fp}")
            continue

        obs = pd.read_csv(fp, encoding="utf-8-sig", low_memory=False)
        # 날짜 보정
        if DATE_COL not in obs.columns and "조사일자(YYYYMMDD)" in obs.columns:
            obs[DATE_COL] = pd.to_datetime(obs["조사일자(YYYYMMDD)"], errors="coerce").dt.normalize()
        else:
            obs[DATE_COL] = pd.to_datetime(obs.get(DATE_COL), errors="coerce").dt.normalize()
        obs = obs[(obs[DATE_COL] >= DATE_START) & (obs[DATE_COL] <= DATE_END)].copy()
        if obs.empty:
            print(f"[스킵] '{name}': 기간 내 관찰 없음")
            continue

        # 이번 병해충에 등장하는 지점ID만 슬라이스
        site_ids = pd.Series(obs["지점ID"].astype(str).dropna().unique())
        neighbors = neighbors_master[neighbors_master["지점ID"].isin(site_ids)]
        if neighbors.empty:
            print(f"[스킵] '{name}': neighbors가 비어 있음(지점ID 매칭 실패)")
            continue

        # 전 기간 base (모든 지점ID × 날짜)
        days = pd.date_range(DATE_START, DATE_END, freq="D")
        base = (pd.DataFrame({"지점ID": site_ids})
                  .assign(key=1)
                  .merge(pd.DataFrame({DATE_COL: days, "key":1}), on="key")
                  .drop(columns="key"))

        agg_cols = ["일강수량", "최고기온", "최대 풍속", "최저기온", "평균 풍속", "평균기온", "DD10", "GDD10_cum"]
        wx_use = wx_all[["지점", DATE_COL] + agg_cols].copy()

        # 결측 보강 기상 계산
        wx_filled = aggregate_weather_by_neighbors(wx_use, neighbors, DATE_COL, agg_cols,
                                                   method=FILL_METHOD, idw_power=IDW_POWER)
        out = base.merge(wx_filled, on=["지점ID", DATE_COL], how="left")

        # 관찰 조인
        pest_cols = pest_related_cols(obs, name)
        keep_extra = [c for c in ALWAYS_KEEP if c in obs.columns]
        obs_keep_cols = ["지점ID", DATE_COL] + pest_cols + keep_extra

        # 품종명은 여러 값이 있어도 first로(필요 시 결합함수로 교체 가능)
        def join_unique_in_order(series, sep="|"):
            vals = pd.unique(series.dropna().astype(str))
            return sep.join(vals)

        agg = {}
        if "품종명" in obs_keep_cols:
            # 여러 품종 그대로 결합하고 싶으면 아래 줄로 바꿔도 됨:
            # agg["품종명"] = join_unique_in_order
            agg["품종명"] = "first"
        for c in pest_cols:
            if c not in agg: agg[c] = "first"

        obs_daily = (obs[obs_keep_cols]
                     .groupby(["지점ID", DATE_COL], as_index=False)
                     .agg(agg) if agg else obs[obs_keep_cols])

        out = out.merge(obs_daily, on=["지점ID", DATE_COL], how="left")

        # === 관측값/발생여부: 우선 (트랩)복숭아순나방-마리수 사용 ===
        val_col = pick_obs_value_col(out, name)
        if val_col:
            out["관측값"] = pd.to_numeric(out[val_col], errors="coerce")
            out["발생여부"] = (out["관측값"].fillna(0) > 0).astype(int)
        else:
            out["관측값"] = np.nan
            out["발생여부"] = np.nan
            print(f"[알림] '{name}': 관측값으로 사용할 컬럼을 찾지 못함 → 관측값/발생여부 NaN")

        # 단위 포함 컬럼 복제
        out["일강수량(mm)"]   = out["일강수량"]
        out["최고기온(°C)"]   = out["최고기온"]
        out["최대 풍속(m/s)"] = out["최대 풍속"]
        out["최저기온(°C)"]   = out["최저기온"]
        out["평균 풍속(m/s)"] = out["평균 풍속"]
        out["평균기온(°C)"]   = out["평균기온"]

        # 컬럼 순서
        front = ["지점ID", DATE_COL, "품종명",
                 "일강수량(mm)", "최고기온(°C)", "최대 풍속(m/s)",
                 "최저기온(°C)", "평균 풍속(m/s)", "평균기온(°C)"]
        pest_cols = [c for c in pest_cols if c not in set(front + ALWAYS_KEEP)]
        tail  = ["DD10", "GDD10_cum", "관측값", "발생여부"]
        cols  = front + pest_cols + tail
        for c in cols:
            if c not in out.columns: out[c] = np.nan

        out = (out.loc[:, cols]
                 .drop_duplicates(subset=["지점ID", DATE_COL])
                 .sort_values(["지점ID", DATE_COL])
                 .reset_index(drop=True))

        save_path = os.path.join(OUT_DIR, f"joined_SAMPLE_GDD_{SPAN_TAG}_{CROP}_{name}.csv")
        out.to_csv(save_path, index=False, encoding="utf-8-sig", float_format="%.1f")
        print(f"[저장] {save_path}  rows={len(out):,}, cols={len(out.columns)}")

if __name__ == "__main__":
    main()


/var/folders/2j/__kxcrvx5ss949vx7sspc6th0000gn/T/ipykernel_4794/1310104472.py:183: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  idx_by_rank = piv.applymap(stn_idx.get).to_numpy(dtype=float)  # NaN 가능
/var/folders/2j/__kxcrvx5ss949vx7sspc6th0000gn/T/ipykernel_4794/1310104472.py:183: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  idx_by_rank = piv.applymap(stn_idx.get).to_numpy(dtype=float)  # NaN 가능
/var/folders/2j/__kxcrvx5ss949vx7sspc6th0000gn/T/ipykernel_4794/1310104472.py:183: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  idx_by_rank = piv.applymap(stn_idx.get).to_numpy(dtype=float)  # NaN 가능
/var/folders/2j/__kxcrvx5ss949vx7sspc6th0000gn/T/ipykernel_4794/1310104472.py:183: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  idx_by_rank = piv.applymap(stn_idx.get).to_numpy(dtype=float)  # NaN 가능
/var/folders/2j/__kxcrvx5ss949vx7sspc6th0000

[저장] /Users/doyoung-gil/연구실/데이터/사과/사과_joined_samples/joined_SAMPLE_GDD_2013_2024_APPLE_복숭아순나방.csv  rows=1,672,906, cols=18


### site별 쪼개기

In [18]:
# split_timeseries_by_site_simple.py
# -*- coding: utf-8 -*-
import os, re, glob
import pandas as pd

IN_DIR   = "/Users/doyoung-gil/연구실/데이터/사과/사과_joined_samples"
OUT_ROOT = "/Users/doyoung-gil/연구실/데이터/사과/사과_joined_samples"
FILE_GLOB = "joined_SAMPLE_GDD_2013_2024_APPLE_복숭아순나방.csv"   # 예: joined_SAMPLE_GDD_1997_2024_RICE_흰등멸구.csv

SITE_COL = "지점ID" 

def sanitize_for_filename(s) -> str:
    """파일명에 쓸 수 있게 간단 정제(공백/특수문자 -> _)"""
    import re
    s = "NA" if pd.isna(s) else str(s).strip()
    return re.sub(r'[^0-9A-Za-z가-힣_.-]+', "_", s)

def parse_crop_name(fname: str):
    """
    파일명: joined_SAMPLE_GDD_{SPAN}_{CROP}_{NAME}.csv
    예시 : joined_SAMPLE_GDD_20221101_20241119_RICE_흰등멸구.csv
    """
    base = os.path.basename(fname)
    m = re.match(r"joined_SAMPLE_GDD_.+?_([A-Za-z]+)_(.+)\.csv$", base)
    if not m:
        return None, None
    crop = m.group(1).upper()
    name = m.group(2)
    return crop, name

def main():
    files = sorted(glob.glob(os.path.join(IN_DIR, FILE_GLOB)))
    if not files:
        raise SystemExit(f"[종료] 입력 없음: {os.path.join(IN_DIR, FILE_GLOB)}")

    for fp in files:
        crop, name = parse_crop_name(fp)
        if crop is None:
            print(f"[스킵] 파일명 패턴 불일치: {fp}")
            continue

        out_dir = os.path.join(OUT_ROOT, f"{name}_by_site")
        os.makedirs(out_dir, exist_ok=True)

        df = pd.read_csv(fp, encoding="utf-8-sig")

        if SITE_COL not in df.columns:
            print(f"[스킵] '{SITE_COL}' 컬럼 없음: {fp}")
            continue

        for site, g in df.groupby(SITE_COL, dropna=False):
            if pd.isna(site):
                continue
            site_str = sanitize_for_filename(site)
            out_path = os.path.join(out_dir, f"{crop}_{name}_site-{site_str}.csv")
            g.to_csv(out_path, index=False, encoding="utf-8-sig")
        print(f"[저장 완료] {name} → {out_dir}")

if __name__ == "__main__":
    main()


[저장 완료] 복숭아순나방 → /Users/doyoung-gil/연구실/데이터/사과/사과_joined_samples/복숭아순나방_by_site


### WeatherMovingAverage 구하기

In [21]:
# make_moving_windows_for_fuzzy.py
from pathlib import Path
import pandas as pd
import numpy as np
import re, os

# ===== 경로/설정 =====
IN_DIR  = Path("/Users/doyoung-gil/연구실/데이터/사과/사과_joined_samples/복숭아순나방_by_site")
OUT_DIR = IN_DIR / "/Users/doyoung-gil/연구실/데이터/사과/사과_joined_samples/WeatherMovingAverage"   # R 스크립트와 동일 폴더명
OUT_DIR.mkdir(parents=True, exist_ok=True)

YEAR_START = 2013
YEAR_END   = 2024
YEARS = list(range(YEAR_START, YEAR_END + 1))
OFFS  = [0, 5, 10, 15, 20, 25]      # 오프셋(일)
WIN   = 30                          # 창 길이(일)
STEP  = 30                          # 창 간격(일) = 30일씩 '차곡차곡'
DAYS_CONST = 30

# 원본 열 이름 매핑
COL_DATE  = "일시"
COL_TMAX  = "최고기온(°C)"
COL_TMIN  = "최저기온(°C)"
COL_TAVG  = "평균기온(°C)"
COL_WMEAN = "평균 풍속(m/s)"
COL_WGUST = "최대 풍속(m/s)"
COL_PRCP  = "일강수량(mm)"
COL_DD10  = "DD10"          # 없거나 전부 NaN이면 tavg 기반으로 계산: max(tavg-10, 0)

def parse_site_id(fname: str) -> str:
    base = os.path.basename(fname)
    m = re.search(r"site-([^./]+)", base)  # 파일명 어딘가의 'site-xxxx' 캡처
    return m.group(1) if m else os.path.splitext(base)[0]

def to_num(s): 
    return pd.to_numeric(s, errors="coerce")

def window_stats(d: pd.DataFrame):
    """30일 창 집계: 평균/최댓값/합계. 보간 없음, NaN은 제외하고 계산."""
    out = {}
    # 평균기온 결측 보정: (tmax+tmin)/2 (둘 다 있을 때만)
    tmax = to_num(d.get(COL_TMAX))
    tmin = to_num(d.get(COL_TMIN))
    tavg = to_num(d.get(COL_TAVG))
    if tavg is None:
        tavg = pd.Series(index=d.index, dtype="float64")
    tavg = tavg.copy()
    fill_mask = (tmax.notna() & tmin.notna() & tavg.isna())
    tavg[fill_mask] = (tmax[fill_mask] + tmin[fill_mask]) / 2.0

    out["tavg_mean_30d"] = float(tavg.mean(skipna=True))
    out["tmin_mean_30d"] = float(tmin.mean(skipna=True))
    out["tmax_mean_30d"] = float(tmax.mean(skipna=True))

    wmean = to_num(d.get(COL_WMEAN))
    wgust = to_num(d.get(COL_WGUST))
    prcp  = to_num(d.get(COL_PRCP))

    out["wind_mean_30d"]     = float(wmean.mean(skipna=True))
    out["wind_gust_max_30d"] = float(wgust.max(skipna=True))
    out["precip_sum_30d"]    = float(prcp.sum(skipna=True))

    dd10 = to_num(d.get(COL_DD10))
    if (dd10 is None) or (isinstance(dd10, pd.Series) and dd10.notna().sum() == 0):
        # DD10 열이 없거나 전부 결측이면 tavg로 대체 계산
        dd10 = np.maximum(tavg - 10.0, 0.0)
    out["gdd10_sum_30d"] = float(pd.Series(dd10).sum(skipna=True))
    return out

def gen_windows_for_year(year: int, offset_days: int):
    """
    해당 연도: 1월 1일 + offset에서 시작해 30일 창을 30일 간격으로 '딱 12개' 생성.
    (마지막 창은 다음 해로 넘어갈 수도 있음: 요구사항대로 그대로 둠)
    """
    base = pd.Timestamp(f"{year}-01-01")
    wins = []
    for i in range(12):
        start = base + pd.Timedelta(days=offset_days + i*STEP)
        end   = start + pd.Timedelta(days=WIN-1)  # 포함 끝
        wins.append((i+1, start, end))
    return wins

def process_one(fp: Path):
    df = pd.read_csv(fp, dtype={"지점ID":"string"}, low_memory=False)
    if COL_DATE not in df.columns:
        print(f"[스킵] 날짜 컬럼 없음: {fp.name}"); 
        return
    df[COL_DATE] = pd.to_datetime(df[COL_DATE], errors="coerce").dt.normalize()
    df = df.sort_values(COL_DATE)

    site_id = parse_site_id(fp.name)
    rows = []

    for yr in YEARS:
        for o in OFFS:
            for idx, start, end in gen_windows_for_year(yr, o):
                win = df[(df[COL_DATE] >= start) & (df[COL_DATE] <= end)]
                stats = (window_stats(win) if not win.empty else
                         {k: np.nan for k in [
                             "tavg_mean_30d","tmin_mean_30d","tmax_mean_30d",
                             "wind_mean_30d","wind_gust_max_30d","precip_sum_30d","gdd10_sum_30d"
                         ]})
                rows.append({
                    "site_id": site_id,
                    "year": yr,
                    "offset_days": o,
                    "window_idx": idx,
                    "start_date": start.date(),
                    "end_date": end.date(),
                    "days": DAYS_CONST,
                    **stats
                })

    out = pd.DataFrame(rows)

    # 최종 컬럼 순서 고정
    COL_ORDER = [
        "site_id","year","offset_days","window_idx",
        "start_date","end_date","days",
        "tavg_mean_30d","tmin_mean_30d","tmax_mean_30d",
        "wind_mean_30d","wind_gust_max_30d","precip_sum_30d","gdd10_sum_30d"
    ]
    site_id = parse_site_id(fp.name)
    out = out[COL_ORDER]

    out_fname = f"site-{site_id}_WeatherMovingAverage.csv"  # ← 여기!
    out_path  = OUT_DIR / out_fname

    out.to_csv(out_path, index=False, encoding="utf-8-sig", float_format="%.3f")
    print(f"[저장] {out_path} rows={len(out):,}")

def main():
    files = sorted(IN_DIR.glob("*site-*.csv"))  # 접두사 유무 상관없이 매칭
    if not files:
        print(f"[종료] 입력 없음: {IN_DIR}")
        return
    for fp in files:
        process_one(fp)

if __name__ == "__main__":
    main()


[저장] /Users/doyoung-gil/연구실/데이터/사과/사과_joined_samples/WeatherMovingAverage/site-30523_63004_WeatherMovingAverage.csv rows=864
[저장] /Users/doyoung-gil/연구실/데이터/사과/사과_joined_samples/WeatherMovingAverage/site-30551_62994_WeatherMovingAverage.csv rows=864
[저장] /Users/doyoung-gil/연구실/데이터/사과/사과_joined_samples/WeatherMovingAverage/site-30636_62943_WeatherMovingAverage.csv rows=864
[저장] /Users/doyoung-gil/연구실/데이터/사과/사과_joined_samples/WeatherMovingAverage/site-30644_62932_WeatherMovingAverage.csv rows=864
[저장] /Users/doyoung-gil/연구실/데이터/사과/사과_joined_samples/WeatherMovingAverage/site-30670_56490_WeatherMovingAverage.csv rows=864
[저장] /Users/doyoung-gil/연구실/데이터/사과/사과_joined_samples/WeatherMovingAverage/site-30672_56490_WeatherMovingAverage.csv rows=864
[저장] /Users/doyoung-gil/연구실/데이터/사과/사과_joined_samples/WeatherMovingAverage/site-30676_56412_WeatherMovingAverage.csv rows=864
[저장] /Users/doyoung-gil/연구실/데이터/사과/사과_joined_samples/WeatherMovingAverage/site-30697_56554_WeatherMovingAverage.csv rows=864


### site-관측소 매핑

In [3]:
# build_neighbors_view_tight.py
# -*- coding: utf-8 -*-
from pathlib import Path
import numpy as np
import pandas as pd

# ===== 설정 =====
K_NEIGH        = 3                         # site당 이웃 관측소 수
MAX_RADIUS_KM  = None                      # 반경 제한 없으면 None, 예: 30
WEATHER_DIR    = Path("/Users/doyoung-gil/연구실/데이터/ASOS+AWS/GDD")
SITE_MASTER    = Path("/Users/doyoung-gil/연구실/데이터/사과/site_master.csv")
OUT_CSV        = Path("/Users/doyoung-gil/연구실/데이터/사과/neighbors_view.csv")
DATE_COL       = "일시"
DATE_START     = pd.Timestamp("2013-01-01")
DATE_END       = pd.Timestamp("2025-01-19")

EARTH_R = 6371000.0  # m

def hv(lat1, lon1, lat2, lon2):
    lat1 = np.radians(lat1); lon1 = np.radians(lon1)
    lat2 = np.radians(lat2); lon2 = np.radians(lon2)
    dlat = lat2 - lat1; dlon = lon2 - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2.0)**2
    return 2 * EARTH_R * np.arcsin(np.sqrt(a))  # meters

def read_stations_tight() -> pd.DataFrame:
    # 필요한 컬럼만 로드
    parts = []
    for y in range(DATE_START.year, DATE_END.year + 1):
        p = WEATHER_DIR / f"ASOS_AWS_{y}_with_GDD10.csv"
        if not p.exists():
            continue
        w = pd.read_csv(p, usecols=["지점","지점명","위도","경도", DATE_COL],
                        encoding="utf-8-sig", low_memory=False)
        w[DATE_COL] = pd.to_datetime(w[DATE_COL], errors="coerce").dt.normalize()
        w = w[(w[DATE_COL] >= DATE_START) & (w[DATE_COL] <= DATE_END)]
        w[["지점","위도","경도"]] = w[["지점","위도","경도"]].apply(
            pd.to_numeric, errors="coerce"
        )
        parts.append(w)
    if not parts:
        raise SystemExit("ASOS_AWS 원천에서 스테이션을 찾을 수 없습니다.")
    stn = (pd.concat(parts, ignore_index=True)
             .dropna(subset=["지점","위도","경도"])
             .drop_duplicates(subset=["지점"])  # 지점 ID 기준 고유화
             .sort_values("지점"))
    # 컬럼 통일
    stn = stn.rename(columns={"지점명":"관측소 지점명", "위도":"관측소 위도", "경도":"관측소 경도"})
    return stn[["지점","관측소 지점명","관측소 위도","관측소 경도"]].reset_index(drop=True)

def main():
    # 1) site master
    site = pd.read_csv(SITE_MASTER, dtype={"지점ID": str})
    site = (site.rename(columns={"지점ID":"site_id","좌표-위도":"site 위도","좌표-경도":"site 경도"})
                 .dropna(subset=["site_id","site 위도","site 경도"]))
    site["site_id"] = site["site_id"].astype(str).str.strip()
    site = site[site["site_id"] != ""].drop_duplicates(subset=["site_id"])

    # 2) stations (tight)
    stn = read_stations_tight()
    stn_lat = stn["관측소 위도"].to_numpy()
    stn_lon = stn["관측소 경도"].to_numpy()

    rows = []
    for sid, slat, slon in site[["site_id","site 위도","site 경도"]].itertuples(index=False):
        d = hv(float(slat), float(slon), stn_lat, stn_lon)  # meters
        if MAX_RADIUS_KM is not None:
            mask = d <= (MAX_RADIUS_KM * 1000.0)
            if not mask.any():   # 반경 내 없으면 전체에서 k-NN
                order = np.argsort(d)[:K_NEIGH]
            else:
                order = np.argsort(d[mask])[:K_NEIGH]
                # mask 인덱스를 원래 인덱스로 복원
                order = np.where(mask)[0][order]
        else:
            order = np.argsort(d)[:K_NEIGH]

        for r, idx in enumerate(order, start=1):
            rows.append({
                "site_id": sid,
                "site 위도": float(slat),
                "site 경도": float(slon),
                "rank" : r,
                "관측소 지점명": stn.at[idx, "관측소 지점명"] if pd.notna(stn.at[idx, "관측소 지점명"]) else "",
                "관측소 위도": float(stn_lat[idx]),
                "관측소 경도": float(stn_lon[idx]),
                "site-관측소 거리": float(d[idx])  # meters
            })

    out = (pd.DataFrame(rows)
             .sort_values(["site_id","rank"])
             .drop_duplicates())  # 동일 거리/중복 방지

    out = out[[
    "site_id", "site 위도", "site 경도",
    "rank",
    "관측소 지점명", "관측소 위도", "관측소 경도",
    "site-관측소 거리"
    ]]
    
    OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    out.to_csv(OUT_CSV, index=False, encoding="utf-8-sig", float_format="%.6f")
    print(f"[저장] {OUT_CSV}  rows={len(out):,}")

if __name__ == "__main__":
    main()


[저장] /Users/doyoung-gil/연구실/데이터/사과/neighbors_view.csv  rows=1,137


### best + 위경도 매핑

In [1]:
# add_and_reorder_coords.py
from pathlib import Path
import pandas as pd

# --- 경로 설정 ---
COORD_CSV = Path("/Users/doyoung-gil/연구실/데이터/관찰데이터+지점ID+datetime/RDA_OBSERVATION_APPLE_with_siteid_dt.csv")
BEST_ROOT  = Path("/Users/doyoung-gil/연구실/데이터/사과/사과_joined_samples/Fuzzy3/best2")

# --- 좌표 마스터: '지점ID'를 'site_id'로 통일 후 매핑 ---
coord = (pd.read_csv(COORD_CSV, encoding="utf-8-sig")
           .rename(columns={"지점ID":"site_id", "좌표-위도":"lat", "좌표-경도":"lon"})
           [["site_id","lat","lon"]]
           .drop_duplicates("site_id")
           .set_index("site_id"))

files = sorted(set(BEST_ROOT.rglob("best_*.csv")) | set(BEST_ROOT.rglob("BEST_*.csv")))
print(f"[발견] {len(files)}개")

for f in files:
    df = pd.read_csv(f, encoding="utf-8-sig")

    # 좌표 붙이기
    m = df.merge(coord, on="site_id", how="left", copy=False)
    m["좌표-위도"] = m["lat"]
    m["좌표-경도"] = m["lon"]
    m.drop(columns=["lat","lon"], inplace=True)

    # site_id 바로 뒤로 재배치 (값/열 삭제 없음)
    cols = list(m.columns)
    cols.remove("좌표-위도")
    cols.remove("좌표-경도")
    pos = cols.index("site_id") + 1
    new_cols = cols[:pos] + ["좌표-위도","좌표-경도"] + cols[pos:]

    m[new_cols].to_csv(f, index=False, encoding="utf-8-sig")
    print(f"[저장] {f} rows={len(m)}")

print("[완료]")


[발견] 379개
[저장] /Users/doyoung-gil/연구실/데이터/사과/사과_joined_samples/Fuzzy3/best2/best_30523_63004.csv rows=12
[저장] /Users/doyoung-gil/연구실/데이터/사과/사과_joined_samples/Fuzzy3/best2/best_30551_62994.csv rows=12
[저장] /Users/doyoung-gil/연구실/데이터/사과/사과_joined_samples/Fuzzy3/best2/best_30636_62943.csv rows=12
[저장] /Users/doyoung-gil/연구실/데이터/사과/사과_joined_samples/Fuzzy3/best2/best_30644_62932.csv rows=12
[저장] /Users/doyoung-gil/연구실/데이터/사과/사과_joined_samples/Fuzzy3/best2/best_30670_56490.csv rows=12
[저장] /Users/doyoung-gil/연구실/데이터/사과/사과_joined_samples/Fuzzy3/best2/best_30672_56490.csv rows=12
[저장] /Users/doyoung-gil/연구실/데이터/사과/사과_joined_samples/Fuzzy3/best2/best_30676_56412.csv rows=12
[저장] /Users/doyoung-gil/연구실/데이터/사과/사과_joined_samples/Fuzzy3/best2/best_30697_56554.csv rows=12
[저장] /Users/doyoung-gil/연구실/데이터/사과/사과_joined_samples/Fuzzy3/best2/best_30945_61933.csv rows=12
[저장] /Users/doyoung-gil/연구실/데이터/사과/사과_joined_samples/Fuzzy3/best2/best_30946_62101.csv rows=12
[저장] /Users/doyoung-gil/연구실/데이터/사과/사과_jo

### 모든 지점별 best파일 합치기 + DOY 변환

In [ ]:
# merge_best_and_dates_to_doy.py
from pathlib import Path
import pandas as pd

# 경로 설정 
BEST_ROOT = Path("/Users/doyoung-gil/연구실/데이터/사과/사과_joined_samples/Fuzzy3/best2")
OUT_CSV   = BEST_ROOT / "best_all_sites_DOY2.csv"

# DOY로 변환할 날짜 컬럼들 (파일에 있는 그대로 이름 사용)
DATE_COLS = [
    "dormancy_start_date",
    "dormancy_end_date",
    "growing_start_date",
    "growing_end_date",
    "flowering_date",
]

# 날짜 -> DOY (1~366, 결측은 NA) 변환
def to_doy(series: pd.Series) -> pd.Series:
    dt = pd.to_datetime(series, errors="coerce", format="%Y-%m-%d")
    return dt.dt.dayofyear.astype("Int64")  # pandas nullable integer

# === 파일 수집 ===
files = sorted(set(BEST_ROOT.rglob("best_*.csv")) | set(BEST_ROOT.rglob("BEST_*.csv")))
print(f"[발견] 대상 파일: {len(files)}개")

# === 읽기 & 합치기 ===
dfs = []
for f in files:
    try:
        df = pd.read_csv(f, encoding="utf-8-sig")
    except UnicodeDecodeError:
        df = pd.read_csv(f, encoding="utf-8")

    # 날짜 컬럼을 DOY로 교체 (존재하는 컬럼만 처리)
    for col in DATE_COLS:
        if col in df.columns:
            df[col] = to_doy(df[col])

    # site_id 바로 뒤에 좌표 배치(이미 있다면 순서만 정리)
    need = ["site_id", "좌표-위도", "좌표-경도"]
    if all(c in df.columns for c in need):
        cols = list(df.columns)
        # 좌표 두 컬럼을 현 위치에서 제거 후 site_id 뒤에 재삽입
        cols.remove("좌표-위도"); cols.remove("좌표-경도")
        pos = cols.index("site_id") + 1
        cols = cols[:pos] + ["좌표-위도", "좌표-경도"] + cols[pos:]
        df = df[cols]

    dfs.append(df)

if not dfs:
    raise SystemExit("[중단] 병합할 파일이 없습니다.")

merged = pd.concat(dfs, ignore_index=True)

# === 저장 ===
merged.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
print(f"[저장] {OUT_CSV}  rows={len(merged)}  cols={len(merged.columns)}")
print("[완료] 모든 날짜 컬럼을 DOY로 변환하여 합본 저장.")


[발견] 대상 파일: 379개
[저장] /Users/doyoung-gil/연구실/데이터/사과/사과_joined_samples/Fuzzy3/best2/best_all_sites_DOY2.csv  rows=4548  cols=14
[완료] 모든 날짜 컬럼을 DOY로 변환하여 합본 저장.


ㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡㅡ

### 30m 기상데이터 편집

In [2]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
/Users/doyoung-gil/Downloads/obs_monthly_normals.csv  →  /Users/doyoung-gil/Downloads/obs_monthly_normals_wide.csv
행: site_id, lon, lat, clim_month
열: tmax, tmin, prcp, srad
"""
from pathlib import Path
import numpy as np
import pandas as pd

# 절대경로 (필요시 여기만 바꾸면 됨)
IN_CSV  = Path("/Users/doyoung-gil/Downloads/obs_monthly_normals.csv")
OUT_CSV = Path("/Users/doyoung-gil/Downloads/obs_monthly_normals_wide.csv")

def load_csv_smart(path: Path) -> pd.DataFrame:
    """한글/UTF-8 등 인코딩 자동 시도."""
    last_err = None
    for enc in ("utf-8-sig", "cp949", "utf-8"):
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception as e:
            last_err = e
    raise RuntimeError(f"CSV 읽기 실패: {path}\n{last_err}")

def main():
    df = load_csv_smart(IN_CSV)

    # 필수 컬럼 확인
    need_cols = {"site_id", "clim_month", "var", "value"}
    if not need_cols.issubset(df.columns):
        raise ValueError(f"입력 CSV에 필수 컬럼 {need_cols} 모두 필요합니다. (현재: {df.columns.tolist()})")

    # 위경도(있으면 사용) 정리: 사이트별 대표 lon/lat 1개만 남기기
    for c in ("lon", "lat"):
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    has_lonlat = {"lon", "lat"}.issubset(df.columns)

    if has_lonlat:
        sites_tbl = (
            df[["site_id", "lon", "lat"]]
            .dropna(subset=["lon", "lat"])
            .drop_duplicates(subset=["site_id"])
        )
    else:
        # 없으면 빈 테이블(병합 시 lon/lat는 NaN)
        sites_tbl = pd.DataFrame(columns=["site_id", "lon", "lat"])

    # 본 데이터 정리
    df = df[["site_id", "clim_month", "var", "value"]].copy()
    df["clim_month"] = pd.to_numeric(df["clim_month"], errors="coerce").astype("Int64")

    # 원하는 변수만
    want = ["tmax", "tmin", "prcp", "srad"]
    df = df[df["var"].isin(want)].copy()

    # 피벗 (중복 있으면 평균)
    wide = (
        df.pivot_table(
            index=["site_id", "clim_month"],
            columns="var",
            values="value",
            aggfunc="mean",
        )
        .reset_index()
    )

    # 모든 지점에 1~12월 강제 생성(없으면 NaN)
    sites = wide["site_id"].dropna().unique()
    months = range(1, 13)
    full = pd.MultiIndex.from_product([sites, months], names=["site_id", "clim_month"])
    wide = (
        wide.set_index(["site_id", "clim_month"])
            .reindex(full)
            .reset_index()
    )

    # 열 순서/누락 채우기
    for col in want:
        if col not in wide.columns:
            wide[col] = np.nan
    wide = wide[["site_id", "clim_month", "tmax", "tmin", "prcp", "srad"]]
    wide = wide.sort_values(["site_id", "clim_month"]).reset_index(drop=True)

    # 위경도 병합(사이트 단위)
    wide = wide.merge(sites_tbl, on="site_id", how="left")
    # 컬럼 순서 최종 정리: site_id, lon, lat, clim_month, …
    wide = wide[["site_id", "lon", "lat", "clim_month", "tmax", "tmin", "prcp", "srad"]]

    # 저장
    OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    wide.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

    # 리포트
    print(f"[OK] saved: {OUT_CSV}")
    print(f" rows: {len(wide):,}")
    print(f" sites: {wide['site_id'].nunique()}")
    print(f" months(unique): {sorted(pd.Series(wide['clim_month']).dropna().unique().tolist())}")
    missing = wide[["tmax","tmin","prcp","srad"]].isna().mean().round(4) * 100
    print(" NaN% by column:")
    print(missing.to_string())

if __name__ == "__main__":
    main()


[OK] saved: /Users/doyoung-gil/Downloads/obs_monthly_normals_wide.csv
 rows: 4,548
 sites: 379
 months(unique): [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
 NaN% by column:
tmax    0.0
tmin    0.0
prcp    0.0
srad    0.0
